# Avaliação de BM25 com BGE-Reranker em datasets BEIR

Este notebook investiga um pipeline de recuperação em duas etapas: o **BM25** faz a recuperação lexical inicial e o **BGE-Reranker** reordena os candidatos com um *cross-encoder*. A avaliação utiliza SciFact, TREC-COVID e NF-Corpus, três coleções do benchmark BEIR com características bastante diferentes [1].

## Objetivo

Avaliar em que condições o reranqueamento neural melhora a qualidade do BM25, qual é seu custo computacional e quais tipos de query mais se beneficiam ou são prejudicados.

## Hipóteses

1. O BGE-Reranker tende a melhorar métricas sensíveis às primeiras posições quando documentos relevantes já estão presentes no conjunto candidato do BM25.
2. O ganho depende do domínio, do perfil das queries e da cobertura do BM25, portanto não deve ser uniforme entre datasets.
3. A melhora de qualidade vem acompanhada de maior latência, o que motiva investigar uma política seletiva para decidir quando rerankear.

## Pipeline experimental

`query → BM25 top-100 → BGE-Reranker → ranking final → nDCG, MRR, Recall e MAP`

Além da comparação principal, o notebook inclui análises de custo, significância estatística, falhas por query, fusão RRF, visualizações qualitativas e classificadores exploratórios para aplicação seletiva do reranker.


### Instalação das dependências

Esta célula instala as bibliotecas necessárias no ambiente do Colab: `ir-datasets` para carregar os datasets BEIR, `rank-bm25` para a recuperação inicial, `transformers` para o reranker, além das bibliotecas de gráficos e embeddings usadas nas visualizações.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install ir-datasets rank-bm25 transformers accelerate pandas tqdm matplotlib seaborn plotly scikit-learn faiss-cpu sentence-transformers scipy


### Imports, configurações e protocolo experimental

Aqui são importadas as bibliotecas usadas no notebook, definido o diretório de cache e configurado o experimento.

Configuração principal:

- recuperação inicial com `BM25Okapi`, usando tokenização lexical simples em minúsculas [5];
- `BM25_TOP_N = 100`: até 100 candidatos são entregues ao reranker;
- avaliação em `k = 5, 10 e 20`, com destaque para as métricas em `k = 10`;
- `BAAI/bge-reranker-base` como *cross-encoder* de reranqueamento [6];
- todas as queries disponíveis nos *splits* selecionados, pois `MAX_QUERIES_PER_DATASET = None`;
- execução em GPU quando CUDA está disponível;
- Dense Retrieval, busca híbrida e *sweep* do número de candidatos permanecem opcionais.

Os tempos medidos são tempos de consulta dentro desta implementação e deste ambiente de execução. Eles servem para comparar os pipelines no experimento, mas não representam um benchmark universal de latência.


In [ ]:
import contextlib
import json
import logging
import math
import os
import re
import shutil
import time
from pathlib import Path

os.environ.setdefault("IR_DATASETS_HOME", "/content/ir_datasets")

import faiss
import ir_datasets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from scipy.stats import wilcoxon
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"

BM25_TOP_N = 100
FINAL_TOP_K = 10
EVAL_K_VALUES = [5, 10, 20]
CANDIDATE_RECALL_N = BM25_TOP_N
BATCH_SIZE = 16

# Use None para avaliar todas as queries. Use um numero menor para teste rapido.
MAX_QUERIES_PER_DATASET = None

# Silencia logs/progresso de download e construção de docstore do ir_datasets.
SILENCE_IR_DATASETS_LOGS = True

# Desative se quiser rodar apenas BM25.
RUN_RERANKER = True

# Opcional: faz uma análise de trade-off variando o número de candidatos do BM25.
RUN_BM25_TOP_N_SWEEP = False
BM25_TOP_N_VALUES = [10, 25, 50, 100, 200]

# Opcional: Dense Retrieval com FAISS e Hybrid BM25 + Dense via RRF.
# Pode ser pesado, especialmente para TREC-COVID.
RUN_DENSE_RETRIEVAL = False
DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DENSE_BATCH_SIZE = 128
DENSE_TOP_K = max(EVAL_K_VALUES)
HYBRID_RRF_K = 60

DATASETS = [
    {
        "name": "SciFact",
        "slug": "scifact",
        "docs_id": "beir/scifact",
        "eval_id": "beir/scifact/test",
    },
    {
        "name": "TREC-COVID",
        "slug": "trec_covid",
        "docs_id": "beir/trec-covid",
        "eval_id": "beir/trec-covid",
    },
    {
        "name": "NF-Corpus",
        "slug": "nfcorpus",
        "docs_id": "beir/nfcorpus",
        "eval_id": "beir/nfcorpus/test",
    },
]

sns.set_theme(style="whitegrid", context="notebook")

logging.getLogger("ir_datasets").setLevel(logging.ERROR)
logging.getLogger("ir_datasets.log").setLevel(logging.ERROR)
logging.getLogger("ir_datasets.util.download").setLevel(logging.ERROR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


### Carregamento dos datasets e recuperação BM25

Esta célula define funções para carregar documentos, queries e qrels de cada dataset BEIR. Ela também cria a tokenização simples, constrói o índice BM25 e implementa a função que recupera os documentos mais bem ranqueados pelo BM25.


In [ ]:
@contextlib.contextmanager
def maybe_silence_ir_datasets():
    if SILENCE_IR_DATASETS_LOGS:
        with open(os.devnull, "w") as devnull:
            with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
                yield
    else:
        yield


def load_beir_dataset(config, max_queries=None):
    with maybe_silence_ir_datasets():
        docs_dataset = ir_datasets.load(config["docs_id"])
        eval_dataset = ir_datasets.load(config["eval_id"])

        documents = {}
        for doc in docs_dataset.docs_iter():
            doc_id = str(doc.doc_id)
            title = getattr(doc, "title", "") or ""
            text = getattr(doc, "text", "") or ""
            documents[doc_id] = f"{title}. {text}".strip()

        queries = {}
        for query in eval_dataset.queries_iter():
            query_id = str(query.query_id)
            queries[query_id] = query.text
            if max_queries is not None and len(queries) >= max_queries:
                break

        selected_query_ids = set(queries.keys())
        qrels = {}
        for qrel in eval_dataset.qrels_iter():
            query_id = str(qrel.query_id)
            if query_id not in selected_query_ids:
                continue

            doc_id = str(qrel.doc_id)
            relevance = int(qrel.relevance)
            qrels.setdefault(query_id, {})[doc_id] = relevance

    queries = {query_id: text for query_id, text in queries.items() if query_id in qrels}
    return documents, queries, qrels


def tokenize(text):
    return re.findall(r"\w+", text.lower())


def build_bm25_index(documents):
    doc_ids = list(documents.keys())
    corpus = [documents[doc_id] for doc_id in doc_ids]
    tokenized_corpus = [tokenize(doc) for doc in tqdm(corpus, desc="Tokenizando corpus")]
    bm25 = BM25Okapi(tokenized_corpus)
    return bm25, doc_ids


def retrieve_bm25(query, bm25, doc_ids, top_k=10):
    scores = bm25.get_scores(tokenize(query))
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [
        {"doc_id": doc_ids[idx], "score": float(scores[idx])}
        for idx in top_indices
    ]


### Datasets avaliados e amostras

As coleções são carregadas por meio do `ir-datasets`, usando as versões disponibilizadas no benchmark BEIR [1].

- **SciFact**: recuperação de evidências científicas para verificar afirmações. A coleção original foi proposta por Wadden et al. [2]. Nesta execução, são avaliadas 300 queries sobre 5.183 documentos.
- **TREC-COVID**: recuperação de literatura científica relacionada à COVID-19, criada durante a pandemia para representar necessidades reais e mutáveis de informação biomédica [3]. Nesta execução, são avaliados 50 tópicos sobre 171.332 documentos.
- **NF-Corpus**: coleção médica que relaciona consultas, em geral curtas, a documentos científicos e informações de saúde [4]. Nesta execução, são avaliadas 323 queries sobre 3.633 documentos.

Os datasets diferem bastante no número de documentos relevantes por query. Isso é importante para interpretar o Recall: TREC-COVID possui centenas de julgamentos relevantes por tópico, enquanto SciFact normalmente possui poucos documentos de evidência.

Para tornar a estrutura concreta, a célula seguinte exibe exemplos de documentos, queries e qrels. O `qrels` contém os julgamentos de relevância usados somente na avaliação, nunca na recuperação.


In [ ]:
def preview_text(text, max_chars=220):
    text = str(text).replace("\n", " ").strip()
    return text if len(text) <= max_chars else text[:max_chars] + "..."


def sample_dataset(config, n_docs=3, n_queries=3, n_qrels=5):
    with maybe_silence_ir_datasets():
        docs_dataset = ir_datasets.load(config["docs_id"])
        eval_dataset = ir_datasets.load(config["eval_id"])

        doc_rows = []
        doc_lookup = {}
        for doc in docs_dataset.docs_iter():
            doc_id = str(doc.doc_id)
            title = getattr(doc, "title", "") or ""
            text = getattr(doc, "text", "") or ""
            full_text = f"{title}. {text}".strip()
            doc_lookup[doc_id] = {
                "title": title,
                "text": text,
                "full_text": full_text,
            }
            if len(doc_rows) < n_docs:
                doc_rows.append({
                    "dataset": config["name"],
                    "doc_id": doc_id,
                    "title": preview_text(title, 120),
                    "text": preview_text(text),
                })

        query_rows = []
        query_lookup = {}
        for query in eval_dataset.queries_iter():
            query_id = str(query.query_id)
            query_lookup[query_id] = query.text
            if len(query_rows) < n_queries:
                query_rows.append({
                    "dataset": config["name"],
                    "query_id": query_id,
                    "query": preview_text(query.text),
                })

        qrel_rows = []
        for qrel in eval_dataset.qrels_iter():
            query_id = str(qrel.query_id)
            doc_id = str(qrel.doc_id)
            doc_info = doc_lookup.get(doc_id, {})
            qrel_rows.append({
                "dataset": config["name"],
                "query_id": query_id,
                "query": preview_text(query_lookup.get(query_id, ""), 180),
                "doc_id": doc_id,
                "relevance": int(qrel.relevance),
                "doc_title": preview_text(doc_info.get("title", ""), 120),
                "doc_preview": preview_text(doc_info.get("full_text", ""), 260),
            })
            if len(qrel_rows) >= n_qrels:
                break

    return (
        pd.DataFrame(doc_rows),
        pd.DataFrame(query_rows),
        pd.DataFrame(qrel_rows),
    )


sample_by_dataset = {}

for config in DATASETS:
    docs_sample, queries_sample, qrels_sample = sample_dataset(config)
    sample_by_dataset[config["slug"]] = {
        "documents": docs_sample,
        "queries": queries_sample,
        "qrels": qrels_sample,
    }

    print("=" * 100)
    print(f"Dataset: {config['name']}")

    print("\nDocumentos:")
    display(docs_sample)

    print("\nQueries:")
    display(queries_sample)

    print("\nQrels:")
    display(qrels_sample)


### Métricas de avaliação

As métricas são calculadas por query e depois agregadas pela média:

- **nDCG@k**: valoriza documentos relevantes nas primeiras posições e considera graus de relevância;
- **MRR@k**: mede o inverso da posição do primeiro documento relevante;
- **Recall@k**: fração dos documentos relevantes da query recuperada nas primeiras `k` posições;
- **MAP@k**: resume a precisão observada a cada documento relevante recuperado até `k`;
- **Candidate Recall@100**: mede a cobertura dos documentos relevantes antes do reranqueamento.

O Candidate Recall representa um limite estrutural do pipeline: o reranker apenas reorganiza os candidatos recuperados pelo BM25 e não consegue recuperar um documento ausente do top-100. Os resultados principais usam `k = 10`, enquanto `k = 5` e `k = 20` ajudam a verificar se a tendência se mantém em diferentes profundidades.


In [ ]:

def dcg_at_k(relevances, k):
    dcg = 0.0
    for i, rel in enumerate(relevances[:k]):
        position = i + 1
        dcg += rel / math.log2(position + 1)
    return dcg


def ndcg_at_k(retrieved_doc_ids, relevant_docs, k):
    relevances = [relevant_docs.get(doc_id, 0) for doc_id in retrieved_doc_ids[:k]]
    dcg = dcg_at_k(relevances, k)
    ideal_relevances = sorted(relevant_docs.values(), reverse=True)
    idcg = dcg_at_k(ideal_relevances, k)
    return 0.0 if idcg == 0 else dcg / idcg


def mrr_at_k(retrieved_doc_ids, relevant_docs, k):
    for i, doc_id in enumerate(retrieved_doc_ids[:k]):
        if relevant_docs.get(doc_id, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def recall_at_k(retrieved_doc_ids, relevant_docs, k):
    relevant_doc_ids = {
        doc_id for doc_id, relevance in relevant_docs.items() if relevance > 0
    }
    if not relevant_doc_ids:
        return 0.0
    retrieved_at_k = set(retrieved_doc_ids[:k])
    return len(retrieved_at_k.intersection(relevant_doc_ids)) / len(relevant_doc_ids)


def average_precision_at_k(retrieved_doc_ids, relevant_docs, k):
    relevant_doc_ids = {
        doc_id for doc_id, relevance in relevant_docs.items() if relevance > 0
    }
    if not relevant_doc_ids:
        return 0.0

    hits = 0
    precision_sum = 0.0
    for i, doc_id in enumerate(retrieved_doc_ids[:k], start=1):
        if doc_id in relevant_doc_ids:
            hits += 1
            precision_sum += hits / i

    return precision_sum / min(len(relevant_doc_ids), k)


def metric_values(retrieved_doc_ids, relevant_docs, k_values=EVAL_K_VALUES):
    values = {}
    for k in k_values:
        values[f"ndcg@{k}"] = ndcg_at_k(retrieved_doc_ids, relevant_docs, k)
        values[f"mrr@{k}"] = mrr_at_k(retrieved_doc_ids, relevant_docs, k)
        values[f"recall@{k}"] = recall_at_k(retrieved_doc_ids, relevant_docs, k)
        values[f"map@{k}"] = average_precision_at_k(retrieved_doc_ids, relevant_docs, k)

    values["ndcg"] = values[f"ndcg@{FINAL_TOP_K}"]
    values["mrr"] = values[f"mrr@{FINAL_TOP_K}"]
    values["recall"] = values[f"recall@{FINAL_TOP_K}"]
    values["map"] = values[f"map@{FINAL_TOP_K}"]
    return values


def timed_retrieve(retrieve_fn, query):
    start = time.perf_counter()
    retrieved = retrieve_fn(query)
    elapsed_sec = time.perf_counter() - start
    return retrieved, elapsed_sec


def ranking_rows(dataset_name, pipeline_name, query_id, query_text, retrieved, relevant_docs, documents):
    rows = []
    for rank, item in enumerate(retrieved, start=1):
        doc_id = item["doc_id"]
        doc_text = documents.get(doc_id, "")
        rows.append({
            "dataset": dataset_name,
            "pipeline": pipeline_name,
            "query_id": query_id,
            "query_text": query_text,
            "rank": rank,
            "doc_id": doc_id,
            "score": item.get("score"),
            "relevance": relevant_docs.get(doc_id, 0),
            "doc_text": doc_text,
            "doc_preview": preview_text(doc_text, 500) if "preview_text" in globals() else str(doc_text)[:500],
        })
    return rows


def grouped_ranking_rows(rankings_df):
    grouped_rows = []
    if rankings_df.empty:
        return pd.DataFrame(grouped_rows)

    sort_cols = ["dataset", "pipeline", "query_id", "rank"]
    rankings_df = rankings_df.sort_values(sort_cols)

    for (dataset, pipeline, query_id), group in rankings_df.groupby(["dataset", "pipeline", "query_id"], sort=False):
        query_text = group["query_text"].iloc[0]
        ranking = []
        for row in group.itertuples(index=False):
            ranking.append({
                "rank": int(row.rank),
                "doc_id": row.doc_id,
                "score": None if pd.isna(row.score) else float(row.score),
                "relevance": int(row.relevance),
                "doc_preview": row.doc_preview,
            })

        grouped_rows.append({
            "dataset": dataset,
            "pipeline": pipeline,
            "query_id": query_id,
            "query_text": query_text,
            "top_k": len(ranking),
            "ranking": ranking,
            "ranking_json": json.dumps(ranking, ensure_ascii=False),
        })

    return pd.DataFrame(grouped_rows)


def save_grouped_rankings_json(df, output_path):
    records = []
    for row in df.itertuples(index=False):
        ranking = row.ranking
        if isinstance(ranking, str):
            ranking = json.loads(ranking)

        records.append({
            "dataset": row.dataset,
            "pipeline": row.pipeline,
            "query_id": str(row.query_id),
            "query_text": row.query_text,
            "top_k": int(row.top_k),
            "ranking": ranking,
        })

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2, allow_nan=False)

    return output_path


def normalize_ranking_for_json(ranking):
    if isinstance(ranking, str):
        ranking = json.loads(ranking)

    normalized = []
    for item in ranking:
        normalized.append({
            "rank": int(item.get("rank")),
            "doc_id": str(item.get("doc_id")),
            "score": None if pd.isna(item.get("score")) else float(item.get("score")),
            "relevance": int(item.get("relevance", 0)),
            "doc_preview": item.get("doc_preview", ""),
        })
    return normalized


def build_bm25_bge_query_rankings(rankings_grouped_df):
    rows = []
    if rankings_grouped_df.empty:
        return rows

    for (dataset, query_id), group in rankings_grouped_df.groupby(["dataset", "query_id"], sort=False):
        bm25_group = group[group["pipeline"] == "BM25"]
        bge_group = group[group["pipeline"] == "BM25 + BGE-Reranker"]

        if bm25_group.empty and bge_group.empty:
            continue

        base_row = bm25_group.iloc[0] if not bm25_group.empty else bge_group.iloc[0]
        ranking_bm25 = [] if bm25_group.empty else normalize_ranking_for_json(bm25_group.iloc[0]["ranking"])
        ranking_bge = [] if bge_group.empty else normalize_ranking_for_json(bge_group.iloc[0]["ranking"])

        rows.append({
            "dataset": dataset,
            "query_id": str(query_id),
            "query": base_row["query_text"],
            "ranking_bm25": ranking_bm25,
            "ranking_bge": ranking_bge,
        })

    return rows


def save_bm25_bge_query_rankings_json(rankings_grouped_df, output_path):
    rows = build_bm25_bge_query_rankings(rankings_grouped_df)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2, allow_nan=False)
    return rows

def evaluate_retrieved(dataset_name, pipeline_name, queries, qrels, retrieve_fn, documents, top_k=10):
    metric_rows = []
    all_ranking_rows = []

    for query_id, query_text in tqdm(queries.items(), desc=f"{dataset_name} | {pipeline_name}"):
        relevant_docs = qrels[query_id]
        retrieved, elapsed_sec = timed_retrieve(retrieve_fn, query_text)
        retrieved_doc_ids = [item["doc_id"] for item in retrieved]
        metrics = metric_values(retrieved_doc_ids, relevant_docs)

        metric_rows.append({
            "dataset": dataset_name,
            "pipeline": pipeline_name,
            "query_id": query_id,
            "query_text": query_text,
            "elapsed_sec": elapsed_sec,
            **metrics,
        })
        all_ranking_rows.extend(
            ranking_rows(dataset_name, pipeline_name, query_id, query_text, retrieved[:top_k], relevant_docs, documents)
        )

    rankings_df = pd.DataFrame(all_ranking_rows)
    grouped_rankings_df = grouped_ranking_rows(rankings_df)
    return pd.DataFrame(metric_rows), rankings_df, grouped_rankings_df


def candidate_recall_at_n(query_text, relevant_docs, bm25, doc_ids, n=100):
    bm25_candidates = retrieve_bm25(query_text, bm25, doc_ids, top_k=n)
    candidate_ids = {item["doc_id"] for item in bm25_candidates}
    relevant_ids = {
        doc_id for doc_id, rel in relevant_docs.items() if rel > 0
    }

    if not relevant_ids:
        return 0.0

    return len(candidate_ids & relevant_ids) / len(relevant_ids)


def evaluate_candidate_recall(dataset_name, queries, qrels, bm25, doc_ids, n=100):
    rows = []
    for query_id, query_text in tqdm(queries.items(), desc=f"{dataset_name} | Candidate Recall@{n}"):
        rows.append({
            "dataset": dataset_name,
            "query_id": query_id,
            f"candidate_recall@{n}": candidate_recall_at_n(
                query_text,
                qrels[query_id],
                bm25,
                doc_ids,
                n=n,
            ),
        })
    return pd.DataFrame(rows)


def summarize_results(df):
    aggregations = {
        "nDCG@10": ("ndcg", "mean"),
        "MRR@10": ("mrr", "mean"),
        "Recall@10": ("recall", "mean"),
        "MAP@10": ("map", "mean"),
        "avg_time_sec": ("elapsed_sec", "mean"),
        "total_time_sec": ("elapsed_sec", "sum"),
        "num_queries": ("query_id", "count"),
    }

    for k in EVAL_K_VALUES:
        aggregations[f"nDCG@{k}"] = (f"ndcg@{k}", "mean")
        aggregations[f"MRR@{k}"] = (f"mrr@{k}", "mean")
        aggregations[f"Recall@{k}"] = (f"recall@{k}", "mean")
        aggregations[f"MAP@{k}"] = (f"map@{k}", "mean")

    return (
        df.groupby(["dataset", "pipeline"], as_index=False)
        .agg(**aggregations)
        .sort_values(["dataset", "pipeline"])
        .reset_index(drop=True)
    )


### Reranker BGE

O modelo `BAAI/bge-reranker-base` é um *cross-encoder* da família BGE [6]. Diferentemente de um recuperador denso que produz embeddings independentes, ele processa cada par `(query, documento)` conjuntamente e gera um escore de relevância.

Neste experimento, o modelo:

- recebe apenas os candidatos encontrados pelo BM25;
- processa textos truncados em até 512 tokens;
- não é ajustado nos datasets avaliados neste notebook;
- substitui a ordem do BM25 pela ordem decrescente dos escores neurais.

Esse desenho costuma melhorar a precisão das primeiras posições, mas exige uma inferência para cada par query-documento e, portanto, aumenta o custo conforme cresce o conjunto candidato.


In [ ]:
def load_reranker(model_name=RERANKER_MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(DEVICE)
    model.eval()
    return tokenizer, model


@torch.no_grad()
def rerank_documents(query, bm25_results, documents, tokenizer, model, batch_size=BATCH_SIZE):
    pairs = []
    doc_ids = []

    for item in bm25_results:
        doc_id = item["doc_id"]
        pairs.append((query, documents[doc_id]))
        doc_ids.append(doc_id)

    all_scores = []
    for start in range(0, len(pairs), batch_size):
        batch_pairs = pairs[start:start + batch_size]
        inputs = tokenizer(
            batch_pairs,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
        outputs = model(**inputs)
        scores = outputs.logits.squeeze(-1).detach().cpu().tolist()
        if isinstance(scores, float):
            scores = [scores]
        all_scores.extend(scores)

    reranked = [
        {"doc_id": doc_id, "score": float(score)}
        for doc_id, score in zip(doc_ids, all_scores)
    ]
    return sorted(reranked, key=lambda x: x["score"], reverse=True)


def retrieve_bm25_plus_reranker(
    query,
    bm25,
    doc_ids,
    documents,
    tokenizer,
    model,
    bm25_top_n=100,
    final_top_k=10,
):
    bm25_candidates = retrieve_bm25(query, bm25, doc_ids, top_k=bm25_top_n)
    reranked_results = rerank_documents(
        query=query,
        bm25_results=bm25_candidates,
        documents=documents,
        tokenizer=tokenizer,
        model=model,
    )
    return reranked_results[:final_top_k]

### Inicialização do reranker

Se `RUN_RERANKER=True`, esta célula baixa/carrega o tokenizer e o modelo BGE-Reranker e move o modelo para GPU quando disponível. Ela também imprime o dispositivo usado na execução.


In [ ]:
tokenizer = None
reranker_model = None

if RUN_RERANKER:
    print("Carregando BGE-Reranker...")
    tokenizer, reranker_model = load_reranker()

print("Device:", DEVICE)

### Execução dos experimentos

Esta é a célula principal do experimento. Para cada dataset, ela carrega os dados, constrói o índice BM25, avalia BM25 puro, avalia BM25 + reranker quando habilitado, salva os CSVs individuais e monta a tabela consolidada de resultados.


In [ ]:

all_results = []
all_rankings = []
all_grouped_rankings = []
candidate_recall_results = []
experiment_cache = {}

for config in DATASETS:
    print("=" * 80)
    print(f"Dataset: {config['name']}")
    print("Carregando dataset...")

    documents, queries, qrels = load_beir_dataset(
        config,
        max_queries=MAX_QUERIES_PER_DATASET,
    )

    print(f"Documentos: {len(documents)}")
    print(f"Queries avaliadas: {len(queries)}")
    print(f"Qrels: {len(qrels)}")

    print("Construindo indice BM25...")
    bm25, doc_ids = build_bm25_index(documents)
    experiment_cache[config["slug"]] = {
        "config": config,
        "documents": documents,
        "queries": queries,
        "qrels": qrels,
        "bm25": bm25,
        "doc_ids": doc_ids,
    }

    candidate_recall_df = evaluate_candidate_recall(
        dataset_name=config["name"],
        queries=queries,
        qrels=qrels,
        bm25=bm25,
        doc_ids=doc_ids,
        n=CANDIDATE_RECALL_N,
    )
    candidate_recall_df.to_csv(f"candidate_recall_{config['slug']}.csv", index=False)
    candidate_recall_results.append(candidate_recall_df)

    bm25_df, bm25_rankings, bm25_grouped_rankings = evaluate_retrieved(
        dataset_name=config["name"],
        pipeline_name="BM25",
        queries=queries,
        qrels=qrels,
        retrieve_fn=lambda query, bm25=bm25, doc_ids=doc_ids: retrieve_bm25(
            query,
            bm25,
            doc_ids,
            top_k=max(EVAL_K_VALUES),
        ),
        documents=documents,
        top_k=max(EVAL_K_VALUES),
    )
    bm25_df.to_csv(f"results_{config['slug']}_bm25.csv", index=False)
    bm25_rankings.to_csv(f"rankings_{config['slug']}_bm25.csv", index=False)
    bm25_grouped_rankings.to_csv(f"rankings_grouped_{config['slug']}_bm25.csv", index=False)
    save_grouped_rankings_json(bm25_grouped_rankings, f"rankings_grouped_{config['slug']}_bm25.json")
    all_results.append(bm25_df)
    all_rankings.append(bm25_rankings)
    all_grouped_rankings.append(bm25_grouped_rankings)

    if RUN_RERANKER:
        reranker_df, reranker_rankings, reranker_grouped_rankings = evaluate_retrieved(
            dataset_name=config["name"],
            pipeline_name="BM25 + BGE-Reranker",
            queries=queries,
            qrels=qrels,
            retrieve_fn=lambda query, bm25=bm25, doc_ids=doc_ids, documents=documents: retrieve_bm25_plus_reranker(
                query=query,
                bm25=bm25,
                doc_ids=doc_ids,
                documents=documents,
                tokenizer=tokenizer,
                model=reranker_model,
                bm25_top_n=BM25_TOP_N,
                final_top_k=max(EVAL_K_VALUES),
            ),
            documents=documents,
            top_k=max(EVAL_K_VALUES),
        )
        reranker_df.to_csv(f"results_{config['slug']}_bm25_bge_reranker.csv", index=False)
        reranker_rankings.to_csv(f"rankings_{config['slug']}_bm25_bge_reranker.csv", index=False)
        reranker_grouped_rankings.to_csv(f"rankings_grouped_{config['slug']}_bm25_bge_reranker.csv", index=False)
        save_grouped_rankings_json(reranker_grouped_rankings, f"rankings_grouped_{config['slug']}_bm25_bge_reranker.json")
        all_results.append(reranker_df)
        all_rankings.append(reranker_rankings)
        all_grouped_rankings.append(reranker_grouped_rankings)

results_all = pd.concat(all_results, ignore_index=True)
rankings_all = pd.concat(all_rankings, ignore_index=True)
rankings_grouped_all = pd.concat(all_grouped_rankings, ignore_index=True)
candidate_recall_all = pd.concat(candidate_recall_results, ignore_index=True)
summary = summarize_results(results_all)

candidate_recall_summary = (
    candidate_recall_all
    .groupby("dataset", as_index=False)[f"candidate_recall@{CANDIDATE_RECALL_N}"]
    .mean()
)

summary = summary.merge(candidate_recall_summary, on="dataset", how="left")

results_all.to_csv("results_all_datasets.csv", index=False)
rankings_all.to_csv("rankings_all_datasets.csv", index=False)
rankings_grouped_all.to_csv("rankings_grouped_all_datasets.csv", index=False)
save_grouped_rankings_json(rankings_grouped_all, "rankings_grouped_all_datasets.json")

if RUN_RERANKER:
    query_rankings_bm25_vs_bge = save_bm25_bge_query_rankings_json(
        rankings_grouped_all,
        "query_rankings_bm25_vs_bge_all_datasets.json",
    )

    for config in DATASETS:
        dataset_grouped_rankings = rankings_grouped_all[
            rankings_grouped_all["dataset"] == config["name"]
        ]
        save_bm25_bge_query_rankings_json(
            dataset_grouped_rankings,
            f"query_rankings_bm25_vs_bge_{config['slug']}.json",
        )
candidate_recall_all.to_csv("candidate_recall_all_datasets.csv", index=False)
summary.to_csv("summary_all_datasets.csv", index=False)

print("Exemplo de ranking agrupado por query:")
display(rankings_grouped_all.head(3))

summary


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Exemplo do JSON BM25 vs BGE por query

Esta célula mostra um exemplo do novo formato pedido: cada objeto contém uma query e os rankings `ranking_bm25` e `ranking_bge` lado a lado.


In [ ]:
if RUN_RERANKER and "query_rankings_bm25_vs_bge" in globals():
    print("Exemplo de query com rankings BM25 e BGE lado a lado:")
    display(pd.DataFrame(query_rankings_bm25_vs_bge[:3]))
else:
    print("Execute a célula principal dos experimentos com RUN_RERANKER=True para gerar query_rankings_bm25_vs_bge.")


## Estatísticas descritivas dos datasets e qrels

Esta seção resume o tamanho e o perfil dos datasets já carregados: quantidade de documentos, queries, tamanho médio dos textos e distribuição de documentos relevantes por query no `qrels`. Isso ajuda a contextualizar os resultados.


In [ ]:
dataset_stats_rows = []
qrels_distribution_rows = []

for slug, case in experiment_cache.items():
    documents = case["documents"]
    queries = case["queries"]
    qrels = case["qrels"]

    doc_lengths = np.array([len(tokenize(text)) for text in documents.values()])
    query_lengths = np.array([len(tokenize(text)) for text in queries.values()])
    relevant_counts = np.array([
        sum(1 for relevance in rels.values() if relevance > 0)
        for rels in qrels.values()
    ])

    dataset_stats_rows.append({
        "dataset": case["config"]["name"],
        "dataset_slug": slug,
        "num_documents": len(documents),
        "num_queries": len(queries),
        "num_qrels_queries": len(qrels),
        "avg_doc_tokens": float(doc_lengths.mean()) if len(doc_lengths) else 0.0,
        "median_doc_tokens": float(np.median(doc_lengths)) if len(doc_lengths) else 0.0,
        "avg_query_tokens": float(query_lengths.mean()) if len(query_lengths) else 0.0,
        "median_query_tokens": float(np.median(query_lengths)) if len(query_lengths) else 0.0,
        "avg_relevant_docs_per_query": float(relevant_counts.mean()) if len(relevant_counts) else 0.0,
        "median_relevant_docs_per_query": float(np.median(relevant_counts)) if len(relevant_counts) else 0.0,
        "max_relevant_docs_per_query": int(relevant_counts.max()) if len(relevant_counts) else 0,
    })

    for query_id, rels in qrels.items():
        qrels_distribution_rows.append({
            "dataset": case["config"]["name"],
            "dataset_slug": slug,
            "query_id": query_id,
            "num_relevant_docs": sum(1 for relevance in rels.values() if relevance > 0),
            "num_judged_docs": len(rels),
        })

dataset_stats = pd.DataFrame(dataset_stats_rows)
qrels_distribution = pd.DataFrame(qrels_distribution_rows)

dataset_stats.to_csv("dataset_descriptive_stats.csv", index=False)
qrels_distribution.to_csv("qrels_distribution_by_query.csv", index=False)

display(dataset_stats)

g = sns.catplot(
    data=qrels_distribution,
    kind="box",
    x="dataset",
    y="num_relevant_docs",
    height=4,
    aspect=1.4,
)
g.set_axis_labels("Dataset", "Docs relevantes por query")
g.figure.suptitle("Distribuição de documentos relevantes no qrels", y=1.03)
g.figure.tight_layout()
g.figure.savefig("qrels_relevance_distribution.png", dpi=160, bbox_inches="tight")
plt.show()


### Tabela comparativa em formato pivotado

Esta célula reorganiza o resumo final para facilitar a comparação lado a lado entre datasets, pipelines e métricas. O resultado é uma tabela com `nDCG@10`, `MRR@10` e `Recall@10` separados por pipeline.


In [ ]:
summary_pivot = summary.pivot(
    index="dataset",
    columns="pipeline",
    values=["nDCG@10", "MRR@10", "Recall@10"],
)
summary_pivot

## OPCIONAL: Dense Retrieval com FAISS e Hybrid Retrieval

Esta seção implementa Dense Retrieval com FAISS e recuperação híbrida BM25+Dense. Ela fica desligada por padrão porque gerar embeddings de todos os documentos pode ser pesado. Para ativar, mude:

```python
RUN_DENSE_RETRIEVAL = True
```


In [ ]:
def build_faiss_dense_index(documents, dense_model, batch_size=DENSE_BATCH_SIZE):
    dense_doc_ids = list(documents.keys())
    doc_texts = [documents[doc_id] for doc_id in dense_doc_ids]
    doc_embeddings = dense_model.encode(
        doc_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")
    dense_index = faiss.IndexFlatIP(doc_embeddings.shape[1])
    dense_index.add(doc_embeddings)
    return dense_index, dense_doc_ids, doc_embeddings


def retrieve_dense_faiss(query, dense_model, dense_index, dense_doc_ids, top_k=10):
    query_embedding = dense_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")
    scores, indices = dense_index.search(query_embedding, top_k)
    return [
        {"doc_id": dense_doc_ids[int(idx)], "score": float(score)}
        for score, idx in zip(scores[0], indices[0])
        if idx != -1
    ]


def reciprocal_rank_fusion(rankings, rrf_k=60, top_k=10):
    fused_scores = {}
    for ranking in rankings:
        for rank, item in enumerate(ranking, start=1):
            doc_id = item["doc_id"]
            fused_scores[doc_id] = fused_scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank)
    fused = [{"doc_id": doc_id, "score": score} for doc_id, score in fused_scores.items()]
    return sorted(fused, key=lambda item: item["score"], reverse=True)[:top_k]


def retrieve_hybrid_rrf(query, bm25, doc_ids, dense_model, dense_index, dense_doc_ids, candidate_n=100, top_k=10):
    bm25_results = retrieve_bm25(query, bm25, doc_ids, top_k=candidate_n)
    dense_results = retrieve_dense_faiss(query, dense_model, dense_index, dense_doc_ids, top_k=candidate_n)
    return reciprocal_rank_fusion([bm25_results, dense_results], rrf_k=HYBRID_RRF_K, top_k=top_k)


def retrieve_dense_plus_reranker(query, dense_model, dense_index, dense_doc_ids, documents, tokenizer, model, dense_top_n=100, final_top_k=10):
    dense_candidates = retrieve_dense_faiss(query, dense_model, dense_index, dense_doc_ids, top_k=dense_top_n)
    return rerank_documents(query, dense_candidates, documents, tokenizer, model)[:final_top_k]


def retrieve_hybrid_plus_reranker(query, bm25, doc_ids, dense_model, dense_index, dense_doc_ids, documents, tokenizer, model, candidate_n=100, final_top_k=10):
    hybrid_candidates = retrieve_hybrid_rrf(query, bm25, doc_ids, dense_model, dense_index, dense_doc_ids, candidate_n=candidate_n, top_k=candidate_n)
    return rerank_documents(query, hybrid_candidates, documents, tokenizer, model)[:final_top_k]


In [ ]:
dense_hybrid_results = []
dense_hybrid_rankings = []
dense_hybrid_grouped_rankings = []
dense_hybrid_summary = pd.DataFrame()

if RUN_DENSE_RETRIEVAL:
    dense_model = SentenceTransformer(DENSE_MODEL_NAME, device=DEVICE)

    for slug, case in experiment_cache.items():
        config = case["config"]
        documents = case["documents"]
        queries = case["queries"]
        qrels = case["qrels"]
        bm25 = case["bm25"]
        doc_ids = case["doc_ids"]

        print("=" * 80)
        print(f"Dense/Hybrid | Dataset: {config['name']}")
        dense_index, dense_doc_ids, dense_doc_embeddings = build_faiss_dense_index(documents, dense_model)

        optional_pipelines = [
            ("Dense Retrieval", lambda query, dense_model=dense_model, dense_index=dense_index, dense_doc_ids=dense_doc_ids: retrieve_dense_faiss(query, dense_model, dense_index, dense_doc_ids, top_k=max(EVAL_K_VALUES)), "dense"),
            ("Hybrid BM25 + Dense", lambda query, bm25=bm25, doc_ids=doc_ids, dense_model=dense_model, dense_index=dense_index, dense_doc_ids=dense_doc_ids: retrieve_hybrid_rrf(query, bm25, doc_ids, dense_model, dense_index, dense_doc_ids, candidate_n=BM25_TOP_N, top_k=max(EVAL_K_VALUES)), "hybrid_rrf"),
        ]

        if RUN_RERANKER:
            optional_pipelines.extend([
                ("Dense Retrieval + BGE-Reranker", lambda query, dense_model=dense_model, dense_index=dense_index, dense_doc_ids=dense_doc_ids, documents=documents: retrieve_dense_plus_reranker(query, dense_model, dense_index, dense_doc_ids, documents, tokenizer, reranker_model, dense_top_n=BM25_TOP_N, final_top_k=max(EVAL_K_VALUES)), "dense_bge_reranker"),
                ("Hybrid BM25 + Dense + BGE-Reranker", lambda query, bm25=bm25, doc_ids=doc_ids, dense_model=dense_model, dense_index=dense_index, dense_doc_ids=dense_doc_ids, documents=documents: retrieve_hybrid_plus_reranker(query, bm25, doc_ids, dense_model, dense_index, dense_doc_ids, documents, tokenizer, reranker_model, candidate_n=BM25_TOP_N, final_top_k=max(EVAL_K_VALUES)), "hybrid_rrf_bge_reranker"),
            ])

        for pipeline_name, retrieve_fn, file_slug in optional_pipelines:
            df, ranking_df, grouped_df = evaluate_retrieved(config["name"], pipeline_name, queries, qrels, retrieve_fn, documents, top_k=max(EVAL_K_VALUES))
            df.to_csv(f"results_{config['slug']}_{file_slug}.csv", index=False)
            ranking_df.to_csv(f"rankings_{config['slug']}_{file_slug}.csv", index=False)
            grouped_df.to_csv(f"rankings_grouped_{config['slug']}_{file_slug}.csv", index=False)
            save_grouped_rankings_json(grouped_df, f"rankings_grouped_{config['slug']}_{file_slug}.json")
            dense_hybrid_results.append(df)
            dense_hybrid_rankings.append(ranking_df)
            dense_hybrid_grouped_rankings.append(grouped_df)

    dense_hybrid_results_all = pd.concat(dense_hybrid_results, ignore_index=True)
    dense_hybrid_rankings_all = pd.concat(dense_hybrid_rankings, ignore_index=True)
    dense_hybrid_grouped_all = pd.concat(dense_hybrid_grouped_rankings, ignore_index=True)
    dense_hybrid_summary = summarize_results(dense_hybrid_results_all)

    dense_hybrid_results_all.to_csv("results_dense_hybrid_all_datasets.csv", index=False)
    dense_hybrid_rankings_all.to_csv("rankings_dense_hybrid_all_datasets.csv", index=False)
    dense_hybrid_grouped_all.to_csv("rankings_grouped_dense_hybrid_all_datasets.csv", index=False)
    save_grouped_rankings_json(dense_hybrid_grouped_all, "rankings_grouped_dense_hybrid_all_datasets.json")
    dense_hybrid_summary.to_csv("summary_dense_hybrid_optional.csv", index=False)
    summary_with_optional_pipelines = pd.concat([summary, dense_hybrid_summary], ignore_index=True)
    summary_with_optional_pipelines.to_csv("summary_with_dense_hybrid_optional.csv", index=False)
    display(summary_with_optional_pipelines)
else:
    dense_hybrid_results_all = pd.DataFrame()
    dense_hybrid_rankings_all = pd.DataFrame()
    dense_hybrid_grouped_all = pd.DataFrame()
    summary_with_optional_pipelines = summary.copy()
    print("Dense/Hybrid desativado. Para rodar, defina RUN_DENSE_RETRIEVAL=True no bloco de configuração.")


## Curvas de Precisão-Revocação

Esta seção gera curvas de precisão-revocação interpolada, comparando os pipelines com base nos rankings salvos e no `qrels`.


In [ ]:
def interpolated_precision_at_recall_levels(retrieved_doc_ids, relevant_docs, recall_levels=None):
    if recall_levels is None:
        recall_levels = np.linspace(0, 1, 11)
    relevant_ids = {str(doc_id) for doc_id, relevance in relevant_docs.items() if relevance > 0}
    if not relevant_ids:
        return {float(level): 0.0 for level in recall_levels}
    precisions = []
    recalls = []
    hits = 0
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if str(doc_id) in relevant_ids:
            hits += 1
        precisions.append(hits / rank)
        recalls.append(hits / len(relevant_ids))
    return {
        float(level): max([p for p, r in zip(precisions, recalls) if r >= level], default=0.0)
        for level in recall_levels
    }


def build_precision_recall_curve_df(rankings_df, experiment_cache, recall_levels=None):
    if recall_levels is None:
        recall_levels = np.linspace(0, 1, 11)
    qrels_by_dataset = {case["config"]["name"]: case["qrels"] for case in experiment_cache.values()}
    rows = []
    if rankings_df.empty:
        return pd.DataFrame(rows)
    for (dataset, pipeline, query_id), group in rankings_df.groupby(["dataset", "pipeline", "query_id"], sort=False):
        group = group.sort_values("rank")
        retrieved_doc_ids = [str(doc_id) for doc_id in group["doc_id"].tolist()]
        relevant_docs = qrels_by_dataset[dataset].get(str(query_id), {})
        values = interpolated_precision_at_recall_levels(retrieved_doc_ids, relevant_docs, recall_levels)
        for recall_level, precision in values.items():
            rows.append({"dataset": dataset, "pipeline": pipeline, "query_id": str(query_id), "recall_level": recall_level, "precision": precision})
    return pd.DataFrame(rows)

combined_rankings_for_pr = rankings_all.copy()
if "dense_hybrid_rankings_all" in globals() and not dense_hybrid_rankings_all.empty:
    combined_rankings_for_pr = pd.concat([combined_rankings_for_pr, dense_hybrid_rankings_all], ignore_index=True)

precision_recall_per_query = build_precision_recall_curve_df(combined_rankings_for_pr, experiment_cache)
precision_recall_curves = precision_recall_per_query.groupby(["dataset", "pipeline", "recall_level"], as_index=False)["precision"].mean()
precision_recall_per_query.to_csv("precision_recall_per_query.csv", index=False)
precision_recall_curves.to_csv("precision_recall_curves.csv", index=False)

if not precision_recall_curves.empty:
    g = sns.relplot(data=precision_recall_curves, kind="line", x="recall_level", y="precision", hue="pipeline", col="dataset", marker="o", height=4, aspect=1.1)
    g.set_axis_labels("Recall", "Precisão interpolada")
    g.set_titles("{col_name}")
    g.figure.suptitle("Curvas de precisão-revocação por dataset", y=1.05)
    g.figure.tight_layout()
    g.figure.savefig("precision_recall_curves.png", dpi=160, bbox_inches="tight")
    plt.show()
    fig = px.line(precision_recall_curves, x="recall_level", y="precision", color="pipeline", facet_col="dataset", markers=True, title="Curvas de precisão-revocação por dataset")
    fig.write_html("precision_recall_curves.html")
    fig.show()

precision_recall_curves.head()


## Análise de trade-off do número de candidatos

Esta seção opcional testa diferentes valores de `BM25_TOP_N` para medir o efeito de entregar mais ou menos candidatos ao reranker. Ela ajuda a responder se aumentar o conjunto inicial melhora a qualidade e quanto isso custa em tempo por query.


In [ ]:
bm25_top_n_sweep_results = []

if RUN_RERANKER and RUN_BM25_TOP_N_SWEEP:
    for slug, case in experiment_cache.items():
        config = case["config"]
        documents = case["documents"]
        queries = case["queries"]
        qrels = case["qrels"]
        bm25 = case["bm25"]
        doc_ids = case["doc_ids"]

        for candidate_n in BM25_TOP_N_VALUES:
            print("=" * 80)
            print(f"Dataset: {config['name']} | BM25_TOP_N={candidate_n}")

            sweep_df, _, _ = evaluate_retrieved(
                dataset_name=config["name"],
                pipeline_name=f"BM25 + BGE-Reranker | topN={candidate_n}",
                queries=queries,
                qrels=qrels,
                retrieve_fn=lambda query, candidate_n=candidate_n, bm25=bm25, doc_ids=doc_ids, documents=documents: retrieve_bm25_plus_reranker(
                    query=query,
                    bm25=bm25,
                    doc_ids=doc_ids,
                    documents=documents,
                    tokenizer=tokenizer,
                    model=reranker_model,
                    bm25_top_n=candidate_n,
                    final_top_k=max(EVAL_K_VALUES),
                ),
                documents=documents,
                top_k=max(EVAL_K_VALUES),
            )
            sweep_summary = summarize_results(sweep_df)
            sweep_summary["BM25_TOP_N"] = candidate_n
            bm25_top_n_sweep_results.append(sweep_summary)

    bm25_top_n_sweep = pd.concat(bm25_top_n_sweep_results, ignore_index=True)
    bm25_top_n_sweep.to_csv("bm25_top_n_sweep.csv", index=False)
    display(bm25_top_n_sweep)
else:
    bm25_top_n_sweep = pd.DataFrame()
    print("Sweep desativado. Para rodar, defina RUN_BM25_TOP_N_SWEEP=True no bloco de configuração.")


In [ ]:
if RUN_RERANKER and RUN_BM25_TOP_N_SWEEP and not bm25_top_n_sweep.empty:
    fig = px.line(
        bm25_top_n_sweep,
        x="BM25_TOP_N",
        y="nDCG@10",
        color="dataset",
        markers=True,
        hover_data=["MRR@10", "Recall@10", "MAP@10", "avg_time_sec"],
        title="Trade-off entre número de candidatos BM25 e nDCG@10 do reranker",
    )
    fig.write_html("bm25_top_n_sweep_ndcg.html")
    fig.show()

    fig = px.line(
        bm25_top_n_sweep,
        x="BM25_TOP_N",
        y="avg_time_sec",
        color="dataset",
        markers=True,
        hover_data=["nDCG@10", "MRR@10", "Recall@10", "MAP@10"],
        title="Custo médio por query ao variar BM25_TOP_N",
    )
    fig.write_html("bm25_top_n_sweep_time.html")
    fig.show()
else:
    print("Sem gráficos de sweep para exibir.")


## Visualização 3D dos candidatos recuperados

Esta seção mostra uma query específica em um espaço 3D aproximado. Os pontos são documentos candidatos, posicionados por embeddings reduzidos com PCA. Para facilitar a leitura, a visualização mostra menos pontos por padrão e prioriza documentos relevantes, top do reranker e chunks que mais subiram ou caíram.

Atenção: o gráfico 3D é uma aproximação visual. Ele ajuda a interpretar comportamento, mas não substitui as métricas.


### Escolha dos datasets e queries para visualização 3D

Aqui você escolhe quais datasets terão visualizações 3D. Por padrão, o notebook gera gráficos para SciFact, TREC-COVID e NF-Corpus. Você também pode fixar uma query específica por dataset usando `VIS_QUERY_IDS`.


In [ ]:
# Escolha quais datasets visualizar.
# Use None para gerar visualizações para todos os datasets em experiment_cache.
VIS_DATASET_SLUGS = ["scifact", "trec_covid", "nfcorpus"]

# Opcional: escolha uma query específica por dataset. None usa a primeira query disponível.
VIS_QUERY_IDS = {
    "scifact": None,
    "trec_covid": None,
    "nfcorpus": None,
}

# Use menos pontos para um 3D mais legível. A recuperação/reranking ainda pode usar mais candidatos.
VIS_NUM_CANDIDATES = 60
VIS_MAX_POINTS_3D = 35
VIS_TOP_RERANKED = 10
VIS_SHARED_TOP_N = 5
VIS_QUERY_LINK_TOP_N = 5
VIS_QRELS_LINK_TOP_N = 5
VIS_LABEL_TOP_N = 8
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

vis_dataset_slugs = VIS_DATASET_SLUGS or list(experiment_cache.keys())
for slug in vis_dataset_slugs:
    case_preview = experiment_cache[slug]
    query_id_preview = VIS_QUERY_IDS.get(slug) or next(iter(case_preview["queries"].keys()))
    print("Dataset:", case_preview["config"]["name"])
    print("Query ID:", query_id_preview)
    print("Query:", case_preview["queries"][query_id_preview])
    print("-" * 80)


### Função para criar gráficos 3D por dataset

Esta célula define uma função que gera, para um dataset e uma query, três visualizações: candidatos em 3D, maiores mudanças de ranking e movimento 3D dos chunks após o rerankeamento. Os arquivos são salvos com o slug do dataset no nome.


In [ ]:
if not RUN_RERANKER:
    raise ValueError("Esta visualizacao precisa de RUN_RERANKER=True.")

color_map = {
    "Top 5 BM25 e BGE": "#d62728",
    "Relevante no qrels": "#2ca02c",
    "Top reranker": "#1f77b4",
    "Subiu": "#17becf",
    "Caiu": "#d62728",
    "Sem mudança": "#9e9e9e",
}

def make_query_link_trace(query_coord, row, trace_name, color, legendgroup, hover_title):
    line_style = dict(color=color, width=4, dash="dash")
    trace_kwargs = dict(
        x=[query_coord[0], row["x"]],
        y=[query_coord[1], row["y"]],
        z=[query_coord[2], row["z"]],
        mode="lines",
        line=line_style,
        opacity=0.78,
        name=trace_name,
        legendgroup=legendgroup,
        hovertemplate=(
            f"<b>{hover_title}</b><br>"
            f"Doc: {row['doc_id']}<br>"
            f"Rank BGE: {int(row['rerank_rank'])}<br>"
            f"Rank BM25: {int(row['bm25_rank'])}<br>"
            f"Relevancia qrels: {int(row['relevance'])}"
            "<extra></extra>"
        ),
    )
    try:
        return go.Scatter3d(**trace_kwargs)
    except ValueError:
        trace_kwargs["line"] = dict(color=color, width=4)
        return go.Scatter3d(**trace_kwargs)


def add_query_to_docs_lines(
    fig,
    query_coord,
    docs_df,
    trace_name,
    color,
    legendgroup,
    hover_title,
):
    for _, row in docs_df.iterrows():
        trace = make_query_link_trace(
            query_coord=query_coord,
            row=row,
            trace_name=trace_name,
            color=color,
            legendgroup=legendgroup,
            hover_title=hover_title,
        )
        trace.showlegend = not any(
            getattr(existing_trace, "legendgroup", None) == legendgroup
            for existing_trace in fig.data
        )
        fig.add_trace(trace)


def add_query_comparison_lines(fig, query_coord, vis_df):
    top_reranker_docs = vis_df.sort_values("rerank_rank").head(VIS_QUERY_LINK_TOP_N)
    top_qrels_docs = (
        vis_df[vis_df["relevance"] > 0]
        .sort_values(["relevance", "rerank_rank", "bm25_rank"], ascending=[False, True, True])
        .head(VIS_QRELS_LINK_TOP_N)
    )

    add_query_to_docs_lines(
        fig=fig,
        query_coord=query_coord,
        docs_df=top_reranker_docs,
        trace_name=f"Query -> top {VIS_QUERY_LINK_TOP_N} BGE",
        color="#111111",
        legendgroup="query_top_bge_links",
        hover_title="Query -> documento top BGE",
    )
    add_query_to_docs_lines(
        fig=fig,
        query_coord=query_coord,
        docs_df=top_qrels_docs,
        trace_name=f"Query -> top {VIS_QRELS_LINK_TOP_N} qrels",
        color="#2ca02c",
        legendgroup="query_top_qrels_links",
        hover_title="Query -> documento relevante no qrels",
    )

def format_query_metrics_html(query_metrics):
    if not query_metrics:
        return ""

    metric_specs = [
        ("nDCG@10", "bm25_ndcg", "reranker_ndcg", "delta_nDCG@10"),
        ("MRR@10", "bm25_mrr", "reranker_mrr", "delta_MRR@10"),
        ("Recall@10", "bm25_recall", "reranker_recall", "delta_Recall@10"),
        ("MAP@10", "bm25_map", "reranker_map", "delta_MAP@10"),
    ]
    parts = []
    for label, bm25_key, bge_key, delta_key in metric_specs:
        if bm25_key not in query_metrics or bge_key not in query_metrics:
            continue
        delta = query_metrics.get(
            delta_key,
            query_metrics[bge_key] - query_metrics[bm25_key],
        )
        parts.append(
            f"{label}: BM25={query_metrics[bm25_key]:.3f} | "
            f"BGE={query_metrics[bge_key]:.3f} | delta={delta:+.3f}"
        )
    return "<br><sup>" + " &nbsp; | &nbsp; ".join(parts) + "</sup>" if parts else ""

def build_3d_visualization_for_dataset(
    dataset_slug,
    query_id=None,
    output_suffix=None,
    show_figures=True,
    generate_overview=True,
    query_metrics=None,
):
    case = experiment_cache[dataset_slug]
    vis_queries = case["queries"]
    vis_query_id = query_id or next(iter(vis_queries.keys()))
    vis_query_text = vis_queries[vis_query_id]
    file_suffix = output_suffix or dataset_slug
    metrics_html = format_query_metrics_html(query_metrics)

    vis_bm25_candidates = retrieve_bm25(
        vis_query_text,
        case["bm25"],
        case["doc_ids"],
        top_k=VIS_NUM_CANDIDATES,
    )

    vis_reranked = rerank_documents(
        query=vis_query_text,
        bm25_results=vis_bm25_candidates,
        documents=case["documents"],
        tokenizer=tokenizer,
        model=reranker_model,
    )

    bm25_rank = {
        item["doc_id"]: rank
        for rank, item in enumerate(vis_bm25_candidates, start=1)
    }
    bm25_score = {item["doc_id"]: item["score"] for item in vis_bm25_candidates}
    rerank_rank = {
        item["doc_id"]: rank
        for rank, item in enumerate(vis_reranked, start=1)
    }
    rerank_score = {item["doc_id"]: item["score"] for item in vis_reranked}

    candidate_doc_ids = [item["doc_id"] for item in vis_bm25_candidates]
    candidate_texts = [case["documents"][doc_id] for doc_id in candidate_doc_ids]
    embedding_texts = [vis_query_text] + candidate_texts
    embeddings = embedding_model.encode(
        embedding_texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    coords = PCA(n_components=3, random_state=42).fit_transform(embeddings)
    query_coord = coords[0]
    doc_coords = coords[1:]

    qrels_for_query = case["qrels"].get(vis_query_id, {})
    vis_rows = []
    for doc_id, coord in zip(candidate_doc_ids, doc_coords):
        before_rank = bm25_rank[doc_id]
        after_rank = rerank_rank[doc_id]
        rank_gain = before_rank - after_rank
        relevance = qrels_for_query.get(doc_id, 0)
        text_preview = preview_text(case["documents"][doc_id], 240)

        if before_rank <= VIS_SHARED_TOP_N and after_rank <= VIS_SHARED_TOP_N:
            visual_group = "Top 5 BM25 e BGE"
        elif relevance > 0:
            visual_group = "Relevante no qrels"
        elif after_rank <= VIS_TOP_RERANKED:
            visual_group = "Top reranker"
        elif rank_gain > 0:
            visual_group = "Subiu"
        elif rank_gain < 0:
            visual_group = "Caiu"
        else:
            visual_group = "Sem mudança"

        vis_rows.append({
            "dataset": case["config"]["name"],
            "dataset_slug": dataset_slug,
            "query_id": vis_query_id,
            "query_text": vis_query_text,
            "doc_id": doc_id,
            "x": coord[0],
            "y": coord[1],
            "z": coord[2],
            "bm25_rank": before_rank,
            "rerank_rank": after_rank,
            "rank_gain": rank_gain,
            "bm25_score": bm25_score[doc_id],
            "reranker_score": rerank_score[doc_id],
            "relevance": relevance,
            "visual_group": visual_group,
            "preview": text_preview,
        })

    vis_df = pd.DataFrame(vis_rows)
    vis_df["abs_rank_gain"] = vis_df["rank_gain"].abs()
    vis_df["label"] = ""

    label_mask = (
        (vis_df["rerank_rank"] <= VIS_LABEL_TOP_N)
        | (vis_df["relevance"] > 0)
        | (vis_df["abs_rank_gain"].rank(method="first", ascending=False) <= VIS_LABEL_TOP_N)
    )
    vis_df.loc[label_mask, "label"] = vis_df.loc[label_mask, "doc_id"].astype(str)

    vis_plot_df = (
        vis_df.assign(
            priority=(vis_df["relevance"] > 0).astype(int) * 1000
            + (VIS_TOP_RERANKED + 1 - vis_df["rerank_rank"].clip(upper=VIS_TOP_RERANKED)).clip(lower=0) * 10
            + vis_df["abs_rank_gain"]
        )
        .sort_values("priority", ascending=False)
        .head(VIS_MAX_POINTS_3D)
        .sort_values("rerank_rank")
    )
    vis_plot_df["marker_size"] = 7 + vis_plot_df["abs_rank_gain"].clip(upper=20) * 0.7

    fig = px.scatter_3d(
        vis_plot_df,
        x="x",
        y="y",
        z="z",
        color="visual_group",
        color_discrete_map=color_map,
        size="marker_size",
        text="label",
        hover_name="doc_id",
        hover_data={
            "bm25_rank": True,
            "rerank_rank": True,
            "rank_gain": True,
            "bm25_score": ":.3f",
            "reranker_score": ":.3f",
            "relevance": True,
            "preview": True,
            "marker_size": False,
            "label": False,
            "x": False,
            "y": False,
            "z": False,
        },
        title=f"Candidatos em 3D - {case['config']['name']} | query {vis_query_id}{metrics_html}",
    )

    fig.add_trace(
        go.Scatter3d(
            x=[query_coord[0]],
            y=[query_coord[1]],
            z=[query_coord[2]],
            mode="markers+text",
            marker=dict(size=11, color="black", symbol="diamond"),
            text=["Query"],
            textposition="top center",
            name="Query",
            hovertemplate=f"<b>Query</b><br>{vis_query_text}<extra></extra>",
        )
    )


    add_query_comparison_lines(fig, query_coord, vis_df)

    fig.update_traces(
        textposition="top center",
        selector=dict(mode="markers+text"),
    )
    fig.update_layout(
        width=980,
        height=720,
        legend_title_text="Tipo de ponto",
        margin=dict(l=0, r=0, t=125 if metrics_html else 70, b=0),
        scene=dict(
            xaxis=dict(title="PCA 1", showbackground=False, showticklabels=False),
            yaxis=dict(title="PCA 2", showbackground=False, showticklabels=False),
            zaxis=dict(title="PCA 3", showbackground=False, showticklabels=False),
            camera=dict(eye=dict(x=1.6, y=1.6, z=1.1)),
        ),
    )

    candidates_html = None
    if generate_overview:
        candidates_html = f"ranking_candidates_3d_{file_suffix}.html"
        fig.write_html(candidates_html)
        if show_figures:
            fig.show()

    rank_change_df = vis_df.sort_values("rank_gain", ascending=False).copy()
    rank_change_df["direction"] = np.select(
        [
            rank_change_df["rank_gain"] > 0,
            rank_change_df["rank_gain"] < 0,
        ],
        ["Subiu", "Caiu"],
        default="Igual",
    )

    rank_change_focus = pd.concat([
        rank_change_df.sort_values("rank_gain", ascending=False).head(15),
        rank_change_df.sort_values("rank_gain", ascending=True).head(15),
    ], ignore_index=True).drop_duplicates("doc_id")

    fig_bar = px.bar(
        rank_change_focus.sort_values("rank_gain", ascending=False),
        x="doc_id",
        y="rank_gain",
        color="direction",
        color_discrete_map={"Subiu": "#17becf", "Caiu": "#d62728", "Igual": "#9e9e9e"},
        hover_data={
            "bm25_rank": True,
            "rerank_rank": True,
            "reranker_score": ":.3f",
            "relevance": True,
            "preview": True,
        },
        title=f"Maiores mudanças de posição - {case['config']['name']} | query {vis_query_id}{metrics_html}",
        labels={
            "doc_id": "Documento",
            "rank_gain": "Ganho de posição: rank BM25 - rank reranker",
        },
    )
    fig_bar.add_hline(y=0, line_width=1, line_color="black")
    fig_bar.update_layout(
        width=980,
        height=560 if metrics_html else 520,
        margin=dict(t=125 if metrics_html else 70),
        xaxis_tickangle=-45,
    )

    rank_changes_html = f"reranking_rank_changes_{file_suffix}.html"
    fig_bar.write_html(rank_changes_html)
    if show_figures:
        fig_bar.show()

    movement_df = vis_df[vis_df["rank_gain"] != 0].copy()
    movement_df = movement_df.sort_values("abs_rank_gain", ascending=False).head(VIS_MAX_POINTS_3D)
    movement_df["direction"] = np.where(movement_df["rank_gain"] > 0, "Subiu", "Caiu")
    movement_df["marker_size"] = 7 + movement_df["abs_rank_gain"].clip(upper=20) * 0.8
    movement_df["label"] = np.where(
        movement_df["abs_rank_gain"].rank(method="first", ascending=False) <= VIS_LABEL_TOP_N,
        movement_df["doc_id"].astype(str),
        "",
    )

    fig_move = px.scatter_3d(
        movement_df,
        x="x",
        y="y",
        z="z",
        color="direction",
        color_discrete_map={"Subiu": "#17becf", "Caiu": "#d62728"},
        size="marker_size",
        text="label",
        hover_name="doc_id",
        hover_data={
            "bm25_rank": True,
            "rerank_rank": True,
            "rank_gain": True,
            "relevance": True,
            "preview": True,
            "marker_size": False,
            "label": False,
            "x": False,
            "y": False,
            "z": False,
        },
        title=f"Movimento dos chunks em 3D - {case['config']['name']} | query {vis_query_id}{metrics_html}",
    )
    fig_move.add_trace(
        go.Scatter3d(
            x=[query_coord[0]],
            y=[query_coord[1]],
            z=[query_coord[2]],
            mode="markers+text",
            marker=dict(size=11, color="black", symbol="diamond"),
            text=["Query"],
            textposition="top center",
            name="Query",
            hovertemplate=f"<b>Query</b><br>{vis_query_text}<extra></extra>",
        )
    )

    add_query_comparison_lines(fig_move, query_coord, vis_df)

    fig_move.update_layout(
        width=980,
        height=720,
        legend_title_text="Movimento",
        margin=dict(l=0, r=0, t=125 if metrics_html else 70, b=0),
        scene=dict(
            xaxis=dict(title="PCA 1", showbackground=False, showticklabels=False),
            yaxis=dict(title="PCA 2", showbackground=False, showticklabels=False),
            zaxis=dict(title="PCA 3", showbackground=False, showticklabels=False),
            camera=dict(eye=dict(x=1.6, y=1.6, z=1.1)),
        ),
    )

    movement_html = f"reranking_movement_3d_{file_suffix}.html"
    fig_move.write_html(movement_html)
    if show_figures:
        fig_move.show()

    vis_df.to_csv(f"visualization_3d_points_{file_suffix}.csv", index=False)
    rank_change_df.to_csv(f"visualization_3d_rank_changes_{file_suffix}.csv", index=False)

    return {
        "dataset_slug": dataset_slug,
        "dataset": case["config"]["name"],
        "query_id": vis_query_id,
        "query_text": vis_query_text,
        "points_df": vis_df,
        "rank_change_df": rank_change_df,
        "files": {
            "candidates_3d": candidates_html,
            "rank_changes": rank_changes_html,
            "movement_3d": movement_html,
            "points_csv": f"visualization_3d_points_{file_suffix}.csv",
            "rank_changes_csv": f"visualization_3d_rank_changes_{file_suffix}.csv",
        },
    }


### Gerar visualizações 3D para todos os datasets escolhidos

Esta célula percorre `VIS_DATASET_SLUGS` e cria arquivos separados para SciFact, TREC-COVID e NF-Corpus. Também salva uma tabela resumo com os nomes dos arquivos gerados.


In [ ]:
visualization_3d_results = []

for dataset_slug in vis_dataset_slugs:
    print("=" * 80)
    print(f"Gerando visualizações 3D para: {dataset_slug}")
    result = build_3d_visualization_for_dataset(
        dataset_slug=dataset_slug,
        query_id=VIS_QUERY_IDS.get(dataset_slug),
        show_figures=True,
    )
    visualization_3d_results.append(result)
    print("Arquivos gerados:")
    for file_path in result["files"].values():
        print("-", file_path)

visualization_3d_summary = pd.DataFrame([
    {
        "dataset_slug": item["dataset_slug"],
        "dataset": item["dataset"],
        "query_id": item["query_id"],
        "query_text": item["query_text"],
        **item["files"],
    }
    for item in visualization_3d_results
])
visualization_3d_summary.to_csv("visualization_3d_summary.csv", index=False)
visualization_3d_summary


### Visualizações 3D dos 3 melhores e 3 piores casos por dataset

Esta célula seleciona, separadamente para cada dataset, as 3 queries com maior melhora e as 3 queries com maior piora pelo `delta_nDCG@10`. Para cada uma delas, gera também a visão geral `ranking_candidates_3d_{dataset}_{caso}_{query}.html`, além dos demais gráficos 3D, e salva arquivos com nomes separados.

Isso permite comparar visualmente os casos em que o reranker mais ajudou e os casos em que ele mais atrapalhou, sem misturar datasets.


In [ ]:
def build_query_deltas(results_df):
    bm25_query = results_df[results_df["pipeline"] == "BM25"][[
        "dataset", "query_id", "ndcg", "mrr", "recall", "map"
    ]].rename(columns={
        "ndcg": "bm25_ndcg",
        "mrr": "bm25_mrr",
        "recall": "bm25_recall",
        "map": "bm25_map",
    })

    reranker_query = results_df[results_df["pipeline"] == "BM25 + BGE-Reranker"][[
        "dataset", "query_id", "ndcg", "mrr", "recall", "map"
    ]].rename(columns={
        "ndcg": "reranker_ndcg",
        "mrr": "reranker_mrr",
        "recall": "reranker_recall",
        "map": "reranker_map",
    })

    deltas = bm25_query.merge(reranker_query, on=["dataset", "query_id"])
    deltas["delta_nDCG@10"] = deltas["reranker_ndcg"] - deltas["bm25_ndcg"]
    deltas["delta_MRR@10"] = deltas["reranker_mrr"] - deltas["bm25_mrr"]
    deltas["delta_Recall@10"] = deltas["reranker_recall"] - deltas["bm25_recall"]
    deltas["delta_MAP@10"] = deltas["reranker_map"] - deltas["bm25_map"]
    return deltas

TOP_BOTTOM_3D_PER_DATASET = 3
GENERATE_TOP_BOTTOM_OVERVIEW = True
SHOW_TOP_BOTTOM_3D_FIGURES = False  # deixe True para exibir as visualizações no notebook

top_bottom_3d_results = []
top_bottom_cases = []

if RUN_RERANKER:
    required_metric_columns = {"bm25_map", "reranker_map", "delta_MAP@10"}
    if (
        "query_deltas" not in globals()
        or not required_metric_columns.issubset(query_deltas.columns)
    ):
        query_deltas = build_query_deltas(results_all)
        query_deltas.to_csv("query_level_metric_deltas.csv", index=False)

    slug_by_dataset = {
        case["config"]["name"]: slug
        for slug, case in experiment_cache.items()
    }

    for dataset, dataset_deltas in query_deltas.groupby("dataset", sort=False):
        dataset_slug = slug_by_dataset[dataset]

        best_cases = (
            dataset_deltas
            .sort_values("delta_nDCG@10", ascending=False)
            .head(TOP_BOTTOM_3D_PER_DATASET)
            .assign(case_type="melhorou")
        )
        worst_cases = (
            dataset_deltas
            .sort_values("delta_nDCG@10", ascending=True)
            .head(TOP_BOTTOM_3D_PER_DATASET)
            .assign(case_type="piorou")
        )

        selected_cases = pd.concat([best_cases, worst_cases], ignore_index=True)

        for _, row in selected_cases.iterrows():
            safe_query_id = re.sub(r"[^A-Za-z0-9_-]+", "_", str(row["query_id"]))
            suffix = f"{dataset_slug}_{row['case_type']}_{safe_query_id}"

            print("=" * 80)
            print(f"Dataset: {dataset} | caso: {row['case_type']} | query: {row['query_id']}")
            print(f"delta_nDCG@10: {row['delta_nDCG@10']:.4f}")
            overview_html = f"ranking_candidates_3d_{suffix}.html"
            if GENERATE_TOP_BOTTOM_OVERVIEW:
                print(f"Visão geral: {overview_html}")

            result = build_3d_visualization_for_dataset(
                dataset_slug=dataset_slug,
                query_id=str(row["query_id"]),
                output_suffix=suffix,
                show_figures=SHOW_TOP_BOTTOM_3D_FIGURES,
                generate_overview=GENERATE_TOP_BOTTOM_OVERVIEW,
                query_metrics=row.to_dict(),
            )

            result_summary = {
                "dataset": dataset,
                "dataset_slug": dataset_slug,
                "case_type": row["case_type"],
                "query_id": str(row["query_id"]),
                "delta_nDCG@10": row["delta_nDCG@10"],
                "delta_MRR@10": row["delta_MRR@10"],
                "delta_Recall@10": row["delta_Recall@10"],
                "bm25_nDCG@10": row["bm25_ndcg"],
                "reranker_nDCG@10": row["reranker_ndcg"],
                "bm25_MRR@10": row["bm25_mrr"],
                "reranker_MRR@10": row["reranker_mrr"],
                "bm25_Recall@10": row["bm25_recall"],
                "reranker_Recall@10": row["reranker_recall"],
                "bm25_MAP@10": row["bm25_map"],
                "reranker_MAP@10": row["reranker_map"],
                "delta_MAP@10": row["delta_MAP@10"],
                "query_text": result["query_text"],
                "overview_3d": result["files"]["candidates_3d"] if GENERATE_TOP_BOTTOM_OVERVIEW else None,
                **result["files"],
            }
            top_bottom_3d_results.append(result_summary)
            top_bottom_cases.append(result)

    top_bottom_3d_summary = pd.DataFrame(top_bottom_3d_results)
    top_bottom_3d_summary.to_csv("visualization_3d_top3_best_worst_summary.csv", index=False)

    with open("visualization_3d_top3_best_worst_summary.json", "w", encoding="utf-8") as f:
        json.dump(
            top_bottom_3d_summary.astype(object).where(pd.notna(top_bottom_3d_summary), None).to_dict(orient="records"),
            f,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )

    display(top_bottom_3d_summary)
else:
    top_bottom_3d_summary = pd.DataFrame()
    print("RUN_RERANKER=False; visualizações 3D de melhores/piores casos indisponíveis.")


## Gráficos de comparação

Os gráficos abaixo apresentam a comparação antes e depois do reranqueamento. O primeiro compara as métricas médias, o segundo mostra o ganho absoluto do reranker e o terceiro exibe a distribuição do ganho por query.

As médias resumem o comportamento geral, mas podem ocultar casos individuais de melhora e piora. Por isso, devem ser lidas em conjunto com os boxplots, o teste pareado e a análise qualitativa por query.


### Gráfico das métricas médias

Esta célula transforma a tabela de resumo para um formato adequado ao `seaborn` e cria gráficos de barras comparando BM25 e BM25 + reranker em cada dataset para `nDCG@10`, `MRR@10` e `Recall@10`.


In [ ]:
metrics_long = summary.melt(
    id_vars=["dataset", "pipeline", "num_queries"],
    value_vars=["nDCG@10", "MRR@10", "Recall@10"],
    var_name="metric",
    value_name="score",
)

g = sns.catplot(
    data=metrics_long,
    kind="bar",
    x="dataset",
    y="score",
    hue="pipeline",
    col="metric",
    sharey=False,
    height=4,
    aspect=1.05,
)
g.set_axis_labels("Dataset", "Score medio")
g.set_titles("{col_name}")
g.set_xticklabels(rotation=25)
g.figure.suptitle("Metricas medias por dataset", y=1.05)
g.figure.tight_layout()
g.figure.savefig("metrics_mean_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

### Ganho médio do reranker

Esta célula calcula a diferença entre BM25 + reranker e BM25 puro em cada métrica. O gráfico resultante mostra o ganho absoluto médio do reranker por dataset, e a tabela salva esses deltas também em CSV.


In [ ]:
if RUN_RERANKER:
    improvement_rows = []
    for dataset in summary["dataset"].unique():
        dataset_summary = summary[summary["dataset"] == dataset].set_index("pipeline")
        for metric in ["nDCG@10", "MRR@10", "Recall@10"]:
            bm25_score = dataset_summary.loc["BM25", metric]
            reranker_score = dataset_summary.loc["BM25 + BGE-Reranker", metric]
            delta_abs = reranker_score - bm25_score
            delta_pct = np.nan if bm25_score == 0 else 100 * delta_abs / bm25_score
            improvement_rows.append({
                "dataset": dataset,
                "metric": metric,
                "BM25": bm25_score,
                "BM25 + BGE-Reranker": reranker_score,
                "delta_abs": delta_abs,
                "delta_pct": delta_pct,
            })

    improvements = pd.DataFrame(improvement_rows)
    improvements.to_csv("metric_improvements.csv", index=False)

    plt.figure(figsize=(11, 5))
    ax = sns.barplot(
        data=improvements,
        x="dataset",
        y="delta_abs",
        hue="metric",
    )
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("Ganho absoluto do reranker sobre BM25")
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Delta do score medio")
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.savefig("metric_improvements_delta.png", dpi=160, bbox_inches="tight")
    plt.show()

    improvements
else:
    print("RUN_RERANKER=False; sem grafico de melhoria pos-rerankeamento.")

### Distribuição do ganho por query

Aqui a comparação é feita query a query. A célula calcula quanto o reranker melhorou ou piorou `nDCG@10`, `MRR@10` e `Recall@10` em cada query e cria boxplots para mostrar a distribuição desses ganhos.


In [ ]:
if RUN_RERANKER:
    if "build_query_deltas" not in globals():
        def build_query_deltas(results_df):
            bm25_query = results_df[results_df["pipeline"] == "BM25"][[
                "dataset", "query_id", "ndcg", "mrr", "recall"
            ]].rename(columns={
                "ndcg": "bm25_ndcg",
                "mrr": "bm25_mrr",
                "recall": "bm25_recall",
            })
            reranker_query = results_df[results_df["pipeline"] == "BM25 + BGE-Reranker"][[
                "dataset", "query_id", "ndcg", "mrr", "recall"
            ]].rename(columns={
                "ndcg": "reranker_ndcg",
                "mrr": "reranker_mrr",
                "recall": "reranker_recall",
            })
            deltas = bm25_query.merge(reranker_query, on=["dataset", "query_id"])
            deltas["delta_nDCG@10"] = deltas["reranker_ndcg"] - deltas["bm25_ndcg"]
            deltas["delta_MRR@10"] = deltas["reranker_mrr"] - deltas["bm25_mrr"]
            deltas["delta_Recall@10"] = deltas["reranker_recall"] - deltas["bm25_recall"]
            return deltas

    query_deltas = build_query_deltas(results_all)
    query_deltas.to_csv("query_level_metric_deltas.csv", index=False)

    query_deltas_long = query_deltas.melt(
        id_vars=["dataset", "query_id"],
        value_vars=["delta_nDCG@10", "delta_MRR@10", "delta_Recall@10"],
        var_name="metric",
        value_name="delta",
    )
    query_deltas_long["metric"] = query_deltas_long["metric"].str.replace("delta_", "", regex=False)

    g = sns.catplot(
        data=query_deltas_long,
        kind="box",
        x="dataset",
        y="delta",
        col="metric",
        sharey=False,
        height=4,
        aspect=1.05,
    )
    for ax in g.axes.flat:
        ax.axhline(0, color="black", linewidth=1)
        ax.tick_params(axis="x", rotation=25)
    g.set_axis_labels("Dataset", "Delta por query")
    g.set_titles("{col_name}")
    g.figure.suptitle("Distribuicao do ganho por query", y=1.05)
    g.figure.tight_layout()
    g.figure.savefig("query_level_delta_distribution.png", dpi=160, bbox_inches="tight")
    plt.show()

    query_deltas_long.head()
else:
    print("RUN_RERANKER=False; sem distribuicao por query.")


## Análise qualitativa separada por dataset

Esta seção analisa o rerankeamento sem misturar datasets. Para cada dataset, ela cria arquivos separados com: análise por query, análise por chunk/documento individual e um JSON consolidado contendo as melhoras e pioras observadas.

A análise por chunk compara a posição do mesmo documento antes e depois do reranker dentro da mesma query. Um `rank_delta` positivo significa que o chunk subiu no ranking após o rerankeamento; um valor negativo significa que ele caiu.


In [ ]:
if RUN_RERANKER:
    def dataframe_to_json_records(df):
        clean_df = df.astype(object).where(pd.notna(df), None)
        return clean_df.to_dict(orient="records")

    query_text_rows = []
    for slug, case in experiment_cache.items():
        for query_id, query_text in case["queries"].items():
            query_text_rows.append({
                "dataset": case["config"]["name"],
                "dataset_slug": slug,
                "query_id": query_id,
                "query": query_text,
            })
    query_text_df = pd.DataFrame(query_text_rows)

    qualitative_queries = query_deltas.merge(
        query_text_df,
        on=["dataset", "query_id"],
        how="left",
    )

    qualitative_queries["interpretacao"] = np.select(
        [
            qualitative_queries["delta_nDCG@10"] > 0,
            qualitative_queries["delta_nDCG@10"] < 0,
        ],
        [
            "reranker melhorou o ranking da query no topo",
            "reranker piorou o ranking da query no topo",
        ],
        default="sem mudança em nDCG@10",
    )

    qualitative_query_json_files = []
    best_queries_by_dataset = []
    worst_queries_by_dataset = []

    for dataset, dataset_queries in qualitative_queries.groupby("dataset", sort=False):
        dataset_slug = dataset_queries["dataset_slug"].iloc[0]
        dataset_queries = dataset_queries.sort_values("delta_nDCG@10", ascending=False)

        best_dataset_queries = dataset_queries.head(10).copy()
        worst_dataset_queries = dataset_queries.tail(10).sort_values("delta_nDCG@10", ascending=True).copy()

        best_queries_by_dataset.append(best_dataset_queries)
        worst_queries_by_dataset.append(worst_dataset_queries)

        query_analysis_payload = {
            "dataset": dataset,
            "dataset_slug": dataset_slug,
            "metric_used_for_sorting": "delta_nDCG@10",
            "most_improved_queries": dataframe_to_json_records(best_dataset_queries),
            "most_worsened_queries": dataframe_to_json_records(worst_dataset_queries),
            "all_query_deltas": dataframe_to_json_records(dataset_queries),
        }

        json_path = f"qualitative_query_analysis_{dataset_slug}.json"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(query_analysis_payload, f, ensure_ascii=False, indent=2, allow_nan=False)
        qualitative_query_json_files.append(json_path)

        dataset_queries.to_csv(f"qualitative_query_deltas_{dataset_slug}.csv", index=False)

    best_queries = pd.concat(best_queries_by_dataset, ignore_index=True)
    worst_queries = pd.concat(worst_queries_by_dataset, ignore_index=True)

    best_queries.to_csv("queries_most_improved_by_dataset.csv", index=False)
    worst_queries.to_csv("queries_most_worsened_by_dataset.csv", index=False)

    print("Arquivos JSON de análise por query salvos:")
    for file_name in qualitative_query_json_files:
        print("-", file_name)

    print("Top queries que mais melhoraram por dataset:")
    display(best_queries[["dataset", "query_id", "query", "bm25_ndcg", "reranker_ndcg", "delta_nDCG@10", "interpretacao"]])

    print("Top queries que mais pioraram por dataset:")
    display(worst_queries[["dataset", "query_id", "query", "bm25_ndcg", "reranker_ndcg", "delta_nDCG@10", "interpretacao"]])
else:
    qualitative_query_json_files = []
    print("RUN_RERANKER=False; análise qualitativa por query indisponível.")


In [ ]:
if RUN_RERANKER:
    if "dataframe_to_json_records" not in globals():
        def dataframe_to_json_records(df):
            clean_df = df.astype(object).where(pd.notna(df), None)
            return clean_df.to_dict(orient="records")

    bm25_chunks = rankings_all[rankings_all["pipeline"] == "BM25"][[
        "dataset", "query_id", "query_text", "doc_id", "rank", "score", "relevance", "doc_preview", "doc_text"
    ]].rename(columns={
        "rank": "bm25_rank",
        "score": "bm25_score",
        "doc_preview": "bm25_doc_preview",
        "doc_text": "bm25_doc_text",
    })

    reranker_chunks = rankings_all[rankings_all["pipeline"] == "BM25 + BGE-Reranker"][[
        "dataset", "query_id", "query_text", "doc_id", "rank", "score", "relevance", "doc_preview", "doc_text"
    ]].rename(columns={
        "rank": "reranker_rank",
        "score": "reranker_score",
        "doc_preview": "reranker_doc_preview",
        "doc_text": "reranker_doc_text",
    })

    chunk_rank_changes = bm25_chunks.merge(
        reranker_chunks,
        on=["dataset", "query_id", "doc_id"],
        how="outer",
        suffixes=("_bm25", "_reranker"),
    )

    max_saved_rank = max(EVAL_K_VALUES)
    missing_rank = max_saved_rank + 1

    chunk_rank_changes["query_text"] = chunk_rank_changes["query_text_bm25"].combine_first(
        chunk_rank_changes["query_text_reranker"]
    )
    chunk_rank_changes["relevance"] = chunk_rank_changes["relevance_bm25"].combine_first(
        chunk_rank_changes["relevance_reranker"]
    ).fillna(0).astype(int)
    chunk_rank_changes["doc_preview"] = chunk_rank_changes["bm25_doc_preview"].combine_first(
        chunk_rank_changes["reranker_doc_preview"]
    )
    chunk_rank_changes["doc_text"] = chunk_rank_changes["bm25_doc_text"].combine_first(
        chunk_rank_changes["reranker_doc_text"]
    )

    chunk_rank_changes["bm25_rank_filled"] = chunk_rank_changes["bm25_rank"].fillna(missing_rank)
    chunk_rank_changes["reranker_rank_filled"] = chunk_rank_changes["reranker_rank"].fillna(missing_rank)
    chunk_rank_changes["rank_delta"] = chunk_rank_changes["bm25_rank_filled"] - chunk_rank_changes["reranker_rank_filled"]

    chunk_rank_changes["movement"] = np.select(
        [
            chunk_rank_changes["rank_delta"] > 0,
            chunk_rank_changes["rank_delta"] < 0,
        ],
        ["melhorou", "piorou"],
        default="igual",
    )

    chunk_rank_changes["appeared_only_after_reranker"] = chunk_rank_changes["bm25_rank"].isna()
    chunk_rank_changes["dropped_after_reranker"] = chunk_rank_changes["reranker_rank"].isna()

    chunk_rank_changes = chunk_rank_changes[[
        "dataset", "query_id", "query_text", "doc_id", "relevance",
        "bm25_rank", "reranker_rank", "rank_delta", "movement",
        "bm25_score", "reranker_score", "appeared_only_after_reranker",
        "dropped_after_reranker", "doc_preview", "doc_text",
    ]].sort_values(["dataset", "query_id", "rank_delta"], ascending=[True, True, False])

    qualitative_chunk_json_files = []
    qualitative_combined_json_files = []
    chunk_analysis_by_dataset = {}

    slug_by_dataset = {
        case["config"]["name"]: slug
        for slug, case in experiment_cache.items()
    }

    for dataset, dataset_chunks in chunk_rank_changes.groupby("dataset", sort=False):
        dataset_slug = slug_by_dataset.get(dataset, dataset.lower().replace(" ", "_"))

        improved_chunks = dataset_chunks[dataset_chunks["rank_delta"] > 0].sort_values(
            "rank_delta", ascending=False
        )
        worsened_chunks = dataset_chunks[dataset_chunks["rank_delta"] < 0].sort_values(
            "rank_delta", ascending=True
        )
        unchanged_chunks = dataset_chunks[dataset_chunks["rank_delta"] == 0]

        chunk_payload = {
            "dataset": dataset,
            "dataset_slug": dataset_slug,
            "rank_delta_definition": "bm25_rank - reranker_rank; positivo = chunk subiu após rerankeamento; negativo = chunk caiu",
            "max_saved_rank": max_saved_rank,
            "num_chunks_analyzed": int(len(dataset_chunks)),
            "num_chunks_improved": int(len(improved_chunks)),
            "num_chunks_worsened": int(len(worsened_chunks)),
            "num_chunks_unchanged": int(len(unchanged_chunks)),
            "improved_chunks": dataframe_to_json_records(improved_chunks),
            "worsened_chunks": dataframe_to_json_records(worsened_chunks),
            "unchanged_chunks": dataframe_to_json_records(unchanged_chunks),
        }

        chunk_json_path = f"qualitative_chunk_analysis_{dataset_slug}.json"
        with open(chunk_json_path, "w", encoding="utf-8") as f:
            json.dump(chunk_payload, f, ensure_ascii=False, indent=2, allow_nan=False)
        qualitative_chunk_json_files.append(chunk_json_path)

        dataset_chunks.to_csv(f"qualitative_chunk_rank_changes_{dataset_slug}.csv", index=False)

        query_payload_path = f"qualitative_query_analysis_{dataset_slug}.json"
        if os.path.exists(query_payload_path):
            with open(query_payload_path, "r", encoding="utf-8") as f:
                query_payload = json.load(f)
        else:
            query_payload = {}

        combined_payload = {
            "dataset": dataset,
            "dataset_slug": dataset_slug,
            "query_level_analysis": query_payload,
            "chunk_level_analysis": chunk_payload,
        }
        combined_json_path = f"qualitative_reranking_analysis_{dataset_slug}.json"
        with open(combined_json_path, "w", encoding="utf-8") as f:
            json.dump(combined_payload, f, ensure_ascii=False, indent=2, allow_nan=False)
        qualitative_combined_json_files.append(combined_json_path)

        chunk_analysis_by_dataset[dataset] = {
            "improved": improved_chunks,
            "worsened": worsened_chunks,
            "unchanged": unchanged_chunks,
        }

    chunk_rank_changes.to_csv("qualitative_chunk_rank_changes_all_datasets.csv", index=False)

    print("Arquivos JSON de análise de chunks salvos:")
    for file_name in qualitative_chunk_json_files:
        print("-", file_name)

    print("Arquivos JSON consolidados por dataset salvos:")
    for file_name in qualitative_combined_json_files:
        print("-", file_name)

    for dataset, movement_groups in chunk_analysis_by_dataset.items():
        print("=" * 80)
        print(f"Dataset: {dataset}")
        print("Chunks individuais que mais melhoraram:")
        display(movement_groups["improved"][[
            "query_id", "query_text", "doc_id", "relevance", "bm25_rank",
            "reranker_rank", "rank_delta", "bm25_score", "reranker_score", "doc_preview"
        ]].head(10))

        print("Chunks individuais que mais pioraram:")
        display(movement_groups["worsened"][[
            "query_id", "query_text", "doc_id", "relevance", "bm25_rank",
            "reranker_rank", "rank_delta", "bm25_score", "reranker_score", "doc_preview"
        ]].head(10))
else:
    qualitative_chunk_json_files = []
    qualitative_combined_json_files = []
    chunk_rank_changes = pd.DataFrame()
    print("RUN_RERANKER=False; análise qualitativa de chunks indisponível.")


## Teste estatístico pareado

Como BM25 e BM25 + BGE-Reranker são avaliados sobre as mesmas queries, aplica-se o teste pareado não paramétrico de Wilcoxon às diferenças por query.

- hipótese nula: a distribuição das diferenças entre os pipelines está centrada em zero;
- nível descritivo adotado: `p < 0,05`;
- a significância deve ser interpretada junto com o tamanho do ganho, a mediana e a distribuição dos deltas.

São realizados vários testes, um por dataset e métrica. Como esta implementação não aplica correção para comparações múltiplas, valores próximos de `0,05` devem ser tratados com cautela. Os resultados mais sólidos são aqueles que combinam ganho relevante, consistência por query e um valor de `p` claramente pequeno.


In [ ]:
if RUN_RERANKER:
    wilcoxon_rows = []
    metric_pairs = [
        ("nDCG@10", "bm25_ndcg", "reranker_ndcg"),
        ("MRR@10", "bm25_mrr", "reranker_mrr"),
        ("Recall@10", "bm25_recall", "reranker_recall"),
    ]

    for dataset, dataset_df in query_deltas.groupby("dataset"):
        for metric_name, bm25_col, reranker_col in metric_pairs:
            diffs = dataset_df[reranker_col] - dataset_df[bm25_col]
            if np.allclose(diffs, 0):
                statistic = 0.0
                p_value = 1.0
            else:
                statistic, p_value = wilcoxon(dataset_df[reranker_col], dataset_df[bm25_col])

            wilcoxon_rows.append({
                "dataset": dataset,
                "metric": metric_name,
                "wilcoxon_statistic": statistic,
                "p_value": p_value,
                "mean_delta": diffs.mean(),
                "median_delta": diffs.median(),
                "significant_at_0_05": p_value < 0.05,
            })

    wilcoxon_results = pd.DataFrame(wilcoxon_rows)
    wilcoxon_results.to_csv("wilcoxon_results.csv", index=False)
    wilcoxon_results
else:
    wilcoxon_results = pd.DataFrame()
    print("RUN_RERANKER=False; teste estatístico indisponível.")


## Análises finais adicionais

Esta seção adiciona oito análises complementares ao experimento:

1. Conclusão automática.
2. Gráficos de qualidade versus custo.
3. Diagnóstico de falha BM25 versus falha do reranker.
4. Heatmap de métricas por dataset/pipeline.
5. Overlap entre top-k do BM25 e do BGE.
6. Análise focada em documentos relevantes que subiram ou caíram.
7. Pipeline pós-hoc de fusão RRF entre BM25 e BGE.
8. Relatório final em Markdown.


In [ ]:
# Utilitários compartilhados das análises finais.
def ensure_query_deltas_available():
    global query_deltas
    if "query_deltas" not in globals():
        query_deltas = build_query_deltas(results_all)
        query_deltas.to_csv("query_level_metric_deltas.csv", index=False)
    return query_deltas


def relevant_ids_for(dataset_name, query_id):
    for case in experiment_cache.values():
        if case["config"]["name"] == dataset_name:
            return {
                str(doc_id)
                for doc_id, relevance in case["qrels"].get(str(query_id), {}).items()
                if relevance > 0
            }
    return set()


def ranked_ids(rankings_df, dataset, pipeline, query_id, k=10):
    rows = rankings_df[
        (rankings_df["dataset"] == dataset)
        & (rankings_df["pipeline"] == pipeline)
        & (rankings_df["query_id"].astype(str) == str(query_id))
        & (rankings_df["rank"] <= k)
    ].sort_values("rank")
    return [str(doc_id) for doc_id in rows["doc_id"].tolist()]


def ranking_items(rankings_df, dataset, pipeline, query_id, k=10):
    rows = rankings_df[
        (rankings_df["dataset"] == dataset)
        & (rankings_df["pipeline"] == pipeline)
        & (rankings_df["query_id"].astype(str) == str(query_id))
        & (rankings_df["rank"] <= k)
    ].sort_values("rank")
    return [
        {"doc_id": str(row.doc_id), "score": float(row.score) if pd.notna(row.score) else 0.0}
        for row in rows.itertuples(index=False)
    ]


def query_text_for(dataset, query_id):
    rows = results_all[
        (results_all["dataset"] == dataset)
        & (results_all["query_id"].astype(str) == str(query_id))
    ]
    if "query_text" in rows.columns and not rows.empty:
        return rows.iloc[0]["query_text"]
    rows = rankings_all[
        (rankings_all["dataset"] == dataset)
        & (rankings_all["query_id"].astype(str) == str(query_id))
    ]
    return "" if rows.empty else rows.iloc[0]["query_text"]


def json_records(df):
    return df.astype(object).where(pd.notna(df), None).to_dict(orient="records")

query_deltas = ensure_query_deltas_available()
analysis_summary_base = summary_with_optional_pipelines if "summary_with_optional_pipelines" in globals() else summary


### 1. Conclusão automática

Esta célula cria uma tabela e um texto curto resumindo, por dataset, se o BGE-Reranker melhorou ou piorou em relação ao BM25, se houve significância estatística e qual foi o custo médio por query.


In [ ]:
automatic_conclusion_rows = []
for dataset in summary["dataset"].unique():
    dataset_summary = summary[summary["dataset"] == dataset].set_index("pipeline")
    if "BM25" not in dataset_summary.index or "BM25 + BGE-Reranker" not in dataset_summary.index:
        continue

    bm25_row = dataset_summary.loc["BM25"]
    bge_row = dataset_summary.loc["BM25 + BGE-Reranker"]
    delta_ndcg = bge_row["nDCG@10"] - bm25_row["nDCG@10"]
    delta_mrr = bge_row["MRR@10"] - bm25_row["MRR@10"]
    delta_recall = bge_row["Recall@10"] - bm25_row["Recall@10"]
    time_ratio = np.nan if bm25_row["avg_time_sec"] == 0 else bge_row["avg_time_sec"] / bm25_row["avg_time_sec"]

    wilcoxon_ndcg = None
    if "wilcoxon_results" in globals() and not wilcoxon_results.empty:
        rows = wilcoxon_results[(wilcoxon_results["dataset"] == dataset) & (wilcoxon_results["metric"] == "nDCG@10")]
        if not rows.empty:
            wilcoxon_ndcg = rows.iloc[0].to_dict()

    automatic_conclusion_rows.append({
        "dataset": dataset,
        "delta_nDCG@10": delta_ndcg,
        "delta_MRR@10": delta_mrr,
        "delta_Recall@10": delta_recall,
        "bm25_avg_time_sec": bm25_row["avg_time_sec"],
        "bge_avg_time_sec": bge_row["avg_time_sec"],
        "time_ratio_bge_vs_bm25": time_ratio,
        "candidate_recall@100": bge_row.get(f"candidate_recall@{CANDIDATE_RECALL_N}", np.nan),
        "wilcoxon_p_value_nDCG@10": None if wilcoxon_ndcg is None else wilcoxon_ndcg["p_value"],
        "wilcoxon_significant_0_05": None if wilcoxon_ndcg is None else bool(wilcoxon_ndcg["significant_at_0_05"]),
        "conclusion": "melhorou" if delta_ndcg > 0 else ("piorou" if delta_ndcg < 0 else "igual"),
    })

automatic_conclusions = pd.DataFrame(automatic_conclusion_rows)
automatic_conclusions.to_csv("automatic_conclusions_by_dataset.csv", index=False)

conclusion_lines = ["# Conclusões automáticas por dataset", ""]
for _, row in automatic_conclusions.iterrows():
    conclusion_lines.append(f"## {row['dataset']}")
    conclusion_lines.append(f"- Resultado principal: **{row['conclusion']}** em nDCG@10.")
    conclusion_lines.append(f"- Delta nDCG@10: {row['delta_nDCG@10']:.4f}.")
    conclusion_lines.append(f"- Delta MRR@10: {row['delta_MRR@10']:.4f}.")
    conclusion_lines.append(f"- Delta Recall@10: {row['delta_Recall@10']:.4f}.")
    conclusion_lines.append(f"- Tempo médio BGE/BM25: {row['time_ratio_bge_vs_bm25']:.2f}x.")
    if row["wilcoxon_p_value_nDCG@10"] is not None:
        conclusion_lines.append(f"- Wilcoxon p-value nDCG@10: {row['wilcoxon_p_value_nDCG@10']:.4g}.")
    conclusion_lines.append("")

with open("automatic_conclusions.md", "w", encoding="utf-8") as f:
    f.write("\n".join(conclusion_lines))

display(automatic_conclusions)


### 2. Qualidade versus custo computacional

Esta célula cria gráficos de trade-off entre qualidade e tempo médio por query.


In [ ]:
quality_cost_df = analysis_summary_base.copy()
quality_cost_df.to_csv("quality_cost_summary.csv", index=False)

for metric in ["nDCG@10", "Recall@10"]:
    plt.figure(figsize=(9, 5))
    ax = sns.scatterplot(
        data=quality_cost_df,
        x="avg_time_sec",
        y=metric,
        hue="pipeline",
        style="dataset",
        s=110,
    )
    ax.set_title(f"{metric} vs tempo médio por query")
    ax.set_xlabel("Tempo médio por query (s)")
    ax.set_ylabel(metric)
    plt.tight_layout()
    file_name = f"quality_cost_{metric.replace('@', '_at_')}.png"
    plt.savefig(file_name, dpi=160, bbox_inches="tight")
    plt.show()

fig = px.scatter(
    quality_cost_df,
    x="avg_time_sec",
    y="nDCG@10",
    color="pipeline",
    symbol="dataset",
    hover_data=["MRR@10", "Recall@10", "MAP@10", "num_queries"],
    title="Trade-off qualidade/custo: nDCG@10 vs tempo médio por query",
)
fig.write_html("quality_cost_ndcg_time.html")
fig.show()


### 3. Diagnóstico de falha: BM25 ou reranker?

Esta célula classifica cada query conforme o comportamento do pipeline: falha de candidatos iniciais, falha do reranker, sucesso do reranker ou caso sem mudança.


In [ ]:
failure_rows = []
for row in query_deltas.itertuples(index=False):
    dataset = row.dataset
    query_id = str(row.query_id)
    relevant_ids = relevant_ids_for(dataset, query_id)
    bm25_top10 = ranked_ids(rankings_all, dataset, "BM25", query_id, k=10)
    bge_top10 = ranked_ids(rankings_all, dataset, "BM25 + BGE-Reranker", query_id, k=10)

    bm25_rel = [doc_id for doc_id in bm25_top10 if doc_id in relevant_ids]
    bge_rel = [doc_id for doc_id in bge_top10 if doc_id in relevant_ids]
    candidate_recall_col = f"candidate_recall@{CANDIDATE_RECALL_N}"
    candidate_rows = candidate_recall_all[
        (candidate_recall_all["dataset"] == dataset)
        & (candidate_recall_all["query_id"].astype(str) == query_id)
    ]
    candidate_recall = None if candidate_rows.empty else float(candidate_rows.iloc[0][candidate_recall_col])

    delta_ndcg = getattr(row, "_6", None)
    # itertuples renames columns with @, so read from query_deltas directly for safety.
    delta_ndcg = float(query_deltas[(query_deltas["dataset"] == dataset) & (query_deltas["query_id"].astype(str) == query_id)].iloc[0]["delta_nDCG@10"])

    if candidate_recall == 0:
        failure_type = "falha_bm25_sem_relevantes_no_top100"
    elif len(bm25_rel) == 0 and len(bge_rel) == 0:
        failure_type = "relevantes_existiam_nos_candidatos_mas_ficaram_fora_top10"
    elif len(bge_rel) < len(bm25_rel) or delta_ndcg < 0:
        failure_type = "falha_ou_piora_do_reranker"
    elif len(bge_rel) > len(bm25_rel) or delta_ndcg > 0:
        failure_type = "sucesso_do_reranker"
    else:
        failure_type = "sem_mudanca_relevante"

    failure_rows.append({
        "dataset": dataset,
        "query_id": query_id,
        "query": query_text_for(dataset, query_id),
        "candidate_recall@100": candidate_recall,
        "num_relevant_qrels": len(relevant_ids),
        "num_bm25_relevant_top10": len(bm25_rel),
        "num_bge_relevant_top10": len(bge_rel),
        "bm25_relevant_top10": bm25_rel,
        "bge_relevant_top10": bge_rel,
        "delta_nDCG@10": delta_ndcg,
        "failure_type": failure_type,
    })

failure_diagnosis = pd.DataFrame(failure_rows)
failure_diagnosis.to_csv("failure_diagnosis_by_query.csv", index=False)

failure_summary = (
    failure_diagnosis
    .groupby(["dataset", "failure_type"], as_index=False)
    .size()
    .rename(columns={"size": "num_queries"})
)
failure_summary.to_csv("failure_diagnosis_summary.csv", index=False)

plt.figure(figsize=(11, 5))
ax = sns.barplot(data=failure_summary, x="dataset", y="num_queries", hue="failure_type")
ax.set_title("Diagnóstico de falhas por dataset")
ax.set_xlabel("Dataset")
ax.set_ylabel("Número de queries")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("failure_diagnosis_summary.png", dpi=160, bbox_inches="tight")
plt.show()

display(failure_summary)


### 4. Heatmap de métricas

Esta célula cria um heatmap para comparar rapidamente datasets, pipelines, métricas e tempo.


In [ ]:
heatmap_metrics = ["nDCG@10", "MRR@10", "Recall@10", "MAP@10", "avg_time_sec"]
heatmap_df = analysis_summary_base.copy()
heatmap_df["dataset_pipeline"] = heatmap_df["dataset"] + " | " + heatmap_df["pipeline"]
heatmap_matrix = heatmap_df.set_index("dataset_pipeline")[heatmap_metrics]

# Normalização por coluna só para visualização: preserva a tabela original nos CSVs.
heatmap_norm = (heatmap_matrix - heatmap_matrix.min()) / (heatmap_matrix.max() - heatmap_matrix.min()).replace(0, 1)

plt.figure(figsize=(10, max(5, len(heatmap_norm) * 0.35)))
ax = sns.heatmap(heatmap_norm, annot=heatmap_matrix.round(4), fmt="", cmap="viridis")
ax.set_title("Heatmap de métricas por dataset/pipeline")
plt.tight_layout()
plt.savefig("metrics_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

heatmap_matrix.to_csv("metrics_heatmap_values.csv")


### 5. Overlap entre top-k do BM25 e do BGE

Esta célula mede quanto do top-10 do BM25 permanece no top-10 após o rerankeamento, e quantos documentos novos aparecem.


In [ ]:
overlap_rows = []
for row in query_deltas.itertuples(index=False):
    dataset = row.dataset
    query_id = str(row.query_id)
    bm25_ids = ranked_ids(rankings_all, dataset, "BM25", query_id, k=10)
    bge_ids = ranked_ids(rankings_all, dataset, "BM25 + BGE-Reranker", query_id, k=10)
    overlap = sorted(set(bm25_ids) & set(bge_ids))
    new_after_bge = [doc_id for doc_id in bge_ids if doc_id not in set(bm25_ids)]
    removed_after_bge = [doc_id for doc_id in bm25_ids if doc_id not in set(bge_ids)]

    overlap_rows.append({
        "dataset": dataset,
        "query_id": query_id,
        "query": query_text_for(dataset, query_id),
        "top10_overlap_count": len(overlap),
        "top10_overlap_ratio": len(overlap) / 10,
        "new_docs_after_bge": new_after_bge,
        "removed_docs_after_bge": removed_after_bge,
    })

topk_overlap = pd.DataFrame(overlap_rows)
topk_overlap.to_csv("top10_overlap_bm25_bge_by_query.csv", index=False)

topk_overlap_summary = (
    topk_overlap
    .groupby("dataset", as_index=False)
    .agg(
        avg_top10_overlap=("top10_overlap_count", "mean"),
        avg_top10_overlap_ratio=("top10_overlap_ratio", "mean"),
    )
)
topk_overlap_summary.to_csv("top10_overlap_bm25_bge_summary.csv", index=False)

plt.figure(figsize=(8, 4))
ax = sns.barplot(data=topk_overlap_summary, x="dataset", y="avg_top10_overlap_ratio")
ax.set_ylim(0, 1)
ax.set_title("Overlap médio entre top-10 BM25 e top-10 BGE")
ax.set_ylabel("Proporção média de overlap")
ax.set_xlabel("Dataset")
plt.tight_layout()
plt.savefig("top10_overlap_bm25_bge.png", dpi=160, bbox_inches="tight")
plt.show()

display(topk_overlap_summary)


### 6. Documentos relevantes que subiram ou caíram

Esta célula filtra a análise de chunks para focar apenas em documentos anotados como relevantes no qrels.


In [ ]:
if "chunk_rank_changes" in globals() and not chunk_rank_changes.empty:
    relevant_rank_movements = chunk_rank_changes[chunk_rank_changes["relevance"] > 0].copy()
else:
    bm25_rel = rankings_all[rankings_all["pipeline"] == "BM25"][["dataset", "query_id", "doc_id", "rank", "score", "relevance", "doc_preview"]].rename(columns={"rank": "bm25_rank", "score": "bm25_score"})
    bge_rel = rankings_all[rankings_all["pipeline"] == "BM25 + BGE-Reranker"][["dataset", "query_id", "doc_id", "rank", "score", "relevance", "doc_preview"]].rename(columns={"rank": "reranker_rank", "score": "reranker_score"})
    relevant_rank_movements = bm25_rel.merge(bge_rel, on=["dataset", "query_id", "doc_id"], how="outer", suffixes=("_bm25", "_bge"))
    relevant_rank_movements["relevance"] = relevant_rank_movements["relevance_bm25"].combine_first(relevant_rank_movements["relevance_bge"]).fillna(0).astype(int)
    relevant_rank_movements["doc_preview"] = relevant_rank_movements["doc_preview_bm25"].combine_first(relevant_rank_movements["doc_preview_bge"])
    missing_rank = max(EVAL_K_VALUES) + 1
    relevant_rank_movements["rank_delta"] = relevant_rank_movements["bm25_rank"].fillna(missing_rank) - relevant_rank_movements["reranker_rank"].fillna(missing_rank)
    relevant_rank_movements = relevant_rank_movements[relevant_rank_movements["relevance"] > 0]

relevant_rank_movements["movement"] = np.select(
    [relevant_rank_movements["rank_delta"] > 0, relevant_rank_movements["rank_delta"] < 0],
    ["subiu", "caiu"],
    default="igual",
)
relevant_rank_movements.to_csv("relevant_documents_rank_movements.csv", index=False)

relevant_movement_summary = (
    relevant_rank_movements
    .groupby(["dataset", "movement"], as_index=False)
    .size()
    .rename(columns={"size": "num_relevant_docs"})
)
relevant_movement_summary.to_csv("relevant_documents_rank_movements_summary.csv", index=False)

plt.figure(figsize=(9, 4))
ax = sns.barplot(data=relevant_movement_summary, x="dataset", y="num_relevant_docs", hue="movement")
ax.set_title("Movimento de documentos relevantes após rerankeamento")
ax.set_xlabel("Dataset")
ax.set_ylabel("Número de documentos relevantes")
plt.tight_layout()
plt.savefig("relevant_documents_rank_movements.png", dpi=160, bbox_inches="tight")
plt.show()

display(relevant_rank_movements.sort_values(["dataset", "rank_delta"], ascending=[True, False]).head(30))


### 7. RRF entre BM25 e BGE

Esta célula cria um pipeline pós-hoc que combina o ranking BM25 e o ranking BGE por Reciprocal Rank Fusion. A ideia é testar se preservar parte do sinal lexical do BM25 reduz pioras do reranker.


In [ ]:
rrf_bm25_bge_rows = []
rrf_bm25_bge_rankings = []

for row in query_deltas.itertuples(index=False):
    dataset = row.dataset
    query_id = str(row.query_id)
    query_text = query_text_for(dataset, query_id)
    relevant_docs = None
    for case in experiment_cache.values():
        if case["config"]["name"] == dataset:
            relevant_docs = case["qrels"].get(query_id, {})
            documents = case["documents"]
            break
    if relevant_docs is None:
        continue

    bm25_items = ranking_items(rankings_all, dataset, "BM25", query_id, k=max(EVAL_K_VALUES))
    bge_items = ranking_items(rankings_all, dataset, "BM25 + BGE-Reranker", query_id, k=max(EVAL_K_VALUES))
    fused = reciprocal_rank_fusion([bm25_items, bge_items], rrf_k=HYBRID_RRF_K, top_k=max(EVAL_K_VALUES))
    fused_doc_ids = [item["doc_id"] for item in fused]
    metrics = metric_values(fused_doc_ids, relevant_docs)

    rrf_bm25_bge_rows.append({
        "dataset": dataset,
        "pipeline": "RRF BM25 + BGE",
        "query_id": query_id,
        "query_text": query_text,
        "elapsed_sec": 0.0,
        **metrics,
    })

    for rank, item in enumerate(fused, start=1):
        doc_id = item["doc_id"]
        rrf_bm25_bge_rankings.append({
            "dataset": dataset,
            "pipeline": "RRF BM25 + BGE",
            "query_id": query_id,
            "query_text": query_text,
            "rank": rank,
            "doc_id": doc_id,
            "score": item["score"],
            "relevance": relevant_docs.get(doc_id, 0),
            "doc_text": documents.get(doc_id, ""),
            "doc_preview": preview_text(documents.get(doc_id, ""), 500),
        })

rrf_bm25_bge_df = pd.DataFrame(rrf_bm25_bge_rows)
rrf_bm25_bge_rankings = pd.DataFrame(rrf_bm25_bge_rankings)
rrf_bm25_bge_grouped = grouped_ranking_rows(rrf_bm25_bge_rankings)
rrf_bm25_bge_summary = summarize_results(rrf_bm25_bge_df)

rrf_bm25_bge_df.to_csv("results_rrf_bm25_bge.csv", index=False)
rrf_bm25_bge_rankings.to_csv("rankings_rrf_bm25_bge.csv", index=False)
rrf_bm25_bge_grouped.to_csv("rankings_grouped_rrf_bm25_bge.csv", index=False)
save_grouped_rankings_json(rrf_bm25_bge_grouped, "rankings_grouped_rrf_bm25_bge.json")
rrf_bm25_bge_summary.to_csv("summary_rrf_bm25_bge.csv", index=False)

summary_with_rrf_bm25_bge = pd.concat([analysis_summary_base, rrf_bm25_bge_summary], ignore_index=True)
summary_with_rrf_bm25_bge.to_csv("summary_with_rrf_bm25_bge.csv", index=False)
display(summary_with_rrf_bm25_bge)


### 8. Relatório final em Markdown

Esta célula exporta um relatório `relatorio_experimento.md` com os principais resultados, gráficos e arquivos gerados.


In [ ]:
from datetime import datetime

report_lines = [
    "# Relatório do experimento BEIR: BM25, BGE-Reranker e análises complementares",
    "",
    f"Gerado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    "## Datasets",
]

for row in dataset_stats.itertuples(index=False):
    report_lines.append(f"- **{row.dataset}**: {row.num_documents} documentos, {row.num_queries} queries, média de {row.avg_relevant_docs_per_query:.2f} docs relevantes por query.")

report_lines.extend(["", "## Resumo principal", ""])
report_lines.append(summary.to_markdown(index=False))

if "automatic_conclusions" in globals() and not automatic_conclusions.empty:
    report_lines.extend(["", "## Conclusões automáticas", ""])
    report_lines.append(automatic_conclusions.to_markdown(index=False))

if "failure_summary" in globals() and not failure_summary.empty:
    report_lines.extend(["", "## Diagnóstico de falhas", ""])
    report_lines.append(failure_summary.to_markdown(index=False))

if "topk_overlap_summary" in globals() and not topk_overlap_summary.empty:
    report_lines.extend(["", "## Overlap BM25 vs BGE", ""])
    report_lines.append(topk_overlap_summary.to_markdown(index=False))

if "rrf_bm25_bge_summary" in globals() and not rrf_bm25_bge_summary.empty:
    report_lines.extend(["", "## RRF BM25 + BGE", ""])
    report_lines.append(rrf_bm25_bge_summary.to_markdown(index=False))

report_lines.extend([
    "",
    "## Figuras principais",
    "",
    "- `metrics_mean_comparison.png`",
    "- `metric_improvements_delta.png`",
    "- `query_level_delta_distribution.png`",
    "- `precision_recall_curves.png`",
    "- `quality_cost_nDCG_at_10.png`",
    "- `quality_cost_Recall_at_10.png`",
    "- `failure_diagnosis_summary.png`",
    "- `metrics_heatmap.png`",
    "- `top10_overlap_bm25_bge.png`",
    "- `relevant_documents_rank_movements.png`",
    "",
    "## Arquivos de auditoria qualitativa",
    "",
    "- `query_rankings_bm25_vs_bge_all_datasets.json`",
    "- `qualitative_query_analysis_<dataset>.json`",
    "- `qualitative_chunk_analysis_<dataset>.json`",
    "- `qualitative_reranking_analysis_<dataset>.json`",
])

with open("relatorio_experimento.md", "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print("Relatório salvo em relatorio_experimento.md")


## Classificador: quando aplicar o reranker?

Esta seção treina uma política seletiva para prever se o BGE-Reranker melhora uma query. O rótulo é `delta_nDCG@10 > 0`; empate e piora são tratados como **não rerankear**.

As features usam somente informações disponíveis antes do rerankeamento: texto da query, distribuição dos scores BM25, margens entre candidatos e cobertura lexical nos documentos recuperados. Qrels e métricas pós-rerankeamento são usados somente para criar o rótulo.

A implementação está inteiramente incorporada ao notebook e não depende de `reranking_query_classifier.py`.


In [ ]:
import json
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

TOKEN_RE = re.compile(r"(?u)\b\w+\b")
TARGET_THRESHOLD = 1e-12


def classifier_tokens(text):
    return TOKEN_RE.findall(str(text).lower())


def safe_ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else 0.0


def softmax_entropy(values):
    shifted = values - values.max()
    probabilities = np.exp(shifted)
    probabilities /= probabilities.sum()
    entropy = -(probabilities * np.log(probabilities + 1e-12)).sum()
    return float(entropy / np.log(len(values))) if len(values) > 1 else 0.0


def query_features(query):
    tokens = classifier_tokens(query)
    char_count = len(query)
    return {
        "query_char_count": char_count,
        "query_word_count": len(tokens),
        "query_unique_ratio": safe_ratio(len(set(tokens)), len(tokens)),
        "query_avg_word_length": safe_ratio(sum(map(len, tokens)), len(tokens)),
        "query_digit_ratio": safe_ratio(sum(c.isdigit() for c in query), char_count),
        "query_punctuation_ratio": safe_ratio(
            sum(not c.isalnum() and not c.isspace() for c in query), char_count
        ),
    }


def ranking_features(group):
    ordered = group.sort_values("rank")
    scores = ordered["score"].astype(float).to_numpy()
    top10_scores = scores[:10]
    query_token_set = set(classifier_tokens(ordered["query_text"].iloc[0]))
    document_token_sets = [
        set(classifier_tokens(text)) for text in ordered["doc_text"].fillna("")
    ]
    coverages = np.array([
        safe_ratio(len(query_token_set & tokens), len(query_token_set))
        for tokens in document_token_sets
    ])
    document_lengths = np.array([len(tokens) for tokens in document_token_sets], dtype=float)
    return {
        "bm25_score_max": scores.max(),
        "bm25_score_mean": scores.mean(),
        "bm25_score_std": scores.std(),
        "bm25_score_range": scores.max() - scores.min(),
        "bm25_top1_top2_margin": scores[0] - scores[1] if len(scores) > 1 else 0.0,
        "bm25_top1_top5_margin": scores[0] - scores[min(4, len(scores) - 1)],
        "bm25_top10_std": top10_scores.std(),
        "bm25_top10_entropy": softmax_entropy(top10_scores),
        "bm25_top1_mean_ratio": safe_ratio(scores[0], scores.mean()),
        "bm25_score_slope": np.polyfit(np.arange(1, len(scores) + 1), scores, 1)[0]
        if len(scores) > 1 else 0.0,
        "lexical_coverage_top1": coverages[0],
        "lexical_coverage_mean": coverages.mean(),
        "lexical_coverage_max": coverages.max(),
        "lexical_coverage_std": coverages.std(),
        "lexical_coverage_top3_mean": coverages[:3].mean(),
        "doc_word_count_mean": document_lengths.mean(),
        "doc_word_count_std": document_lengths.std(),
    }


def build_pre_reranking_features(rankings):
    required = {"dataset", "pipeline", "query_id", "query_text", "rank", "score", "doc_text"}
    missing = required - set(rankings.columns)
    if missing:
        raise ValueError(f"Colunas ausentes no ranking: {sorted(missing)}")
    rows = []
    bm25 = rankings[rankings["pipeline"] == "BM25"]
    for (dataset, query_id), group in bm25.groupby(["dataset", "query_id"], sort=False):
        query = str(group["query_text"].iloc[0])
        rows.append({
            "dataset": dataset,
            "query_id": str(query_id),
            "query_text": query,
            **query_features(query),
            **ranking_features(group),
        })
    return pd.DataFrame(rows)


def build_classifier_dataset(rankings, metric_deltas):
    features = build_pre_reranking_features(rankings)
    labels = metric_deltas[["dataset", "query_id", "delta_nDCG@10"]].copy()
    labels["query_id"] = labels["query_id"].astype(str)
    data = features.merge(labels, on=["dataset", "query_id"], validate="one_to_one")
    data["rerank_beneficial"] = (data["delta_nDCG@10"] > TARGET_THRESHOLD).astype(int)
    return data


def make_query_classifier(numeric_columns):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    features = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        ("query_tfidf", TfidfVectorizer(
            lowercase=True, ngram_range=(1, 2), min_df=2,
            max_features=3000, sublinear_tf=True,
        ), "query_text"),
    ])
    return Pipeline([
        ("features", features),
        ("classifier", LogisticRegression(
            class_weight="balanced", max_iter=2000, random_state=42,
        )),
    ])


def classifier_evaluation_summary(frame, probabilities, evaluation):
    target = frame["rerank_beneficial"].to_numpy()
    predicted = (probabilities >= 0.5).astype(int)
    deltas = frame["delta_nDCG@10"].to_numpy()
    selected_delta = deltas * predicted
    positive_gain = np.clip(deltas, 0, None)
    return {
        "evaluation": evaluation,
        "queries": len(frame),
        "positive_rate": target.mean(),
        "roc_auc": roc_auc_score(target, probabilities),
        "average_precision": average_precision_score(target, probabilities),
        "balanced_accuracy": balanced_accuracy_score(target, predicted),
        "precision": precision_score(target, predicted, zero_division=0),
        "recall": recall_score(target, predicted, zero_division=0),
        "f1": f1_score(target, predicted, zero_division=0),
        "rerank_rate": predicted.mean(),
        "mean_delta_always_rerank": deltas.mean(),
        "mean_delta_classifier": selected_delta.mean(),
        "mean_delta_oracle": positive_gain.mean(),
        "gain_retention_vs_always": safe_ratio(selected_delta.mean(), deltas.mean()),
        "reranker_calls_saved_ratio": 1.0 - predicted.mean(),
        "captured_positive_gain_ratio": safe_ratio(
            (positive_gain * predicted).sum(), positive_gain.sum()
        ),
        "harmful_queries_avoided_ratio": safe_ratio(
            ((deltas < -TARGET_THRESHOLD) & (predicted == 0)).sum(),
            (deltas < -TARGET_THRESHOLD).sum(),
        ),
    }


def evaluate_query_classifier(frame):
    excluded = {"dataset", "query_id", "query_text", "delta_nDCG@10", "rerank_beneficial"}
    numeric_columns = [column for column in frame.columns if column not in excluded]
    x = frame[["query_text", *numeric_columns]]
    y = frame["rerank_beneficial"]
    evaluations = [
        ("stratified_5_fold", StratifiedKFold(5, shuffle=True, random_state=42), None),
        ("leave_one_dataset_out", GroupKFold(frame["dataset"].nunique()), frame["dataset"]),
    ]
    summaries = []
    prediction_frames = []
    for name, cv, groups in evaluations:
        probabilities = cross_val_predict(
            make_query_classifier(numeric_columns), x, y, groups=groups,
            cv=cv, method="predict_proba", n_jobs=-1,
        )[:, 1]
        predictions = frame[[
            "dataset", "query_id", "query_text", "delta_nDCG@10", "rerank_beneficial"
        ]].copy()
        predictions["evaluation"] = name
        predictions["rerank_probability"] = probabilities
        predictions["rerank_predicted"] = (probabilities >= 0.5).astype(int)
        prediction_frames.append(predictions)
        summaries.append(classifier_evaluation_summary(frame, probabilities, name))
    return pd.DataFrame(summaries), pd.concat(prediction_frames, ignore_index=True)


def predict_reranking_need(rankings, model, threshold=0.5):
    features = build_pre_reranking_features(rankings)
    probabilities = model.predict_proba(features[list(model.feature_names_in_)])[:, 1]
    result = features[["dataset", "query_id", "query_text"]].copy()
    result["rerank_probability"] = probabilities
    result["rerank_predicted"] = (probabilities >= threshold).astype(int)
    return result


def run_query_classifier_experiment(rankings, metric_deltas, output_dir="."):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    data = build_classifier_dataset(rankings, metric_deltas)
    summary, predictions = evaluate_query_classifier(data)
    excluded = {"dataset", "query_id", "query_text", "delta_nDCG@10", "rerank_beneficial"}
    numeric_columns = [column for column in data.columns if column not in excluded]
    model = make_query_classifier(numeric_columns)
    model.fit(data[["query_text", *numeric_columns]], data["rerank_beneficial"])
    feature_names = model.named_steps["features"].get_feature_names_out()
    coefficients = model.named_steps["classifier"].coef_[0]
    coefficient_table = pd.DataFrame({
        "feature": feature_names, "coefficient": coefficients,
    }).sort_values("coefficient", key=lambda values: values.abs(), ascending=False)
    data.to_csv(output_dir / "query_classifier_dataset.csv", index=False)
    summary.to_csv(output_dir / "query_classifier_evaluation.csv", index=False)
    predictions.to_csv(output_dir / "query_classifier_predictions.csv", index=False)
    coefficient_table.to_csv(output_dir / "query_classifier_coefficients.csv", index=False)
    joblib.dump(model, output_dir / "query_reranking_classifier.joblib")
    metadata = {
        "target": "delta_nDCG@10 > 0",
        "decision_threshold": 0.5,
        "numeric_features": numeric_columns,
        "important_constraint": "Todas as features existem antes do rerankeamento.",
    }
    (output_dir / "query_classifier_metadata.json").write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    return {"dataset": data, "summary": summary, "predictions": predictions, "model": model}


In [ ]:
query_classifier_results = run_query_classifier_experiment(
    rankings=rankings_all,
    metric_deltas=query_deltas,
    output_dir=".",
)

display(query_classifier_results["summary"].round(4))
display(
    query_classifier_results["dataset"]
    .groupby("dataset")["rerank_beneficial"]
    .agg(queries="size", beneficial_rate="mean")
    .reset_index()
    .round(4)
)


## Benchmark query-only

### Comparação de modelos para decidir quando rerankear

Os modelos abaixo recebem **somente o texto da query**. Scores, posições e documentos do BM25 não são usados como features. O rótulo continua sendo `delta_nDCG@10 > 0`.

São comparados TF-IDF de palavras, TF-IDF de caracteres, *gradient boosting* com atributos linguísticos, embedding com regressão logística e embedding com uma pequena rede neural MLP.

Esta análise é exploratória. A validação estratificada mede generalização entre queries misturando os três datasets, enquanto *leave-one-dataset-out* é um cenário mais rigoroso de transferência para um domínio não visto.


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

QUERY_ONLY_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
QUERY_ONLY_SEED = 42


def build_query_only_data(rankings, metric_deltas):
    queries = rankings[["dataset", "query_id", "query_text"]].drop_duplicates(
        ["dataset", "query_id"]
    ).copy()
    queries["query_id"] = queries["query_id"].astype(str)
    labels = metric_deltas[["dataset", "query_id", "delta_nDCG@10"]].copy()
    labels["query_id"] = labels["query_id"].astype(str)
    data = queries.merge(labels, on=["dataset", "query_id"], validate="one_to_one")
    data["rerank_beneficial"] = (data["delta_nDCG@10"] > TARGET_THRESHOLD).astype(int)
    linguistic = pd.DataFrame(
        [query_features(str(query)) for query in data["query_text"]],
        index=data.index,
    )
    return pd.concat([data, linguistic], axis=1)


def query_only_model_specs():
    balanced_logistic = lambda: LogisticRegression(
        class_weight="balanced", max_iter=2000, random_state=QUERY_ONLY_SEED
    )
    return {
        "word_tfidf_logistic": ("text", Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2), min_df=2, max_features=5000, sublinear_tf=True
            )),
            ("classifier", balanced_logistic()),
        ]), False),
        "char_tfidf_logistic": ("text", Pipeline([
            ("tfidf", TfidfVectorizer(
                analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                max_features=8000, sublinear_tf=True,
            )),
            ("classifier", balanced_logistic()),
        ]), False),
        "linguistic_gradient_boosting": ("linguistic", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", HistGradientBoostingClassifier(
                learning_rate=0.05, max_iter=150, max_leaf_nodes=15,
                l2_regularization=1.0, class_weight="balanced",
                random_state=QUERY_ONLY_SEED,
            )),
        ]), False),
        "embedding_logistic": ("embedding", Pipeline([
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(
                class_weight="balanced", C=0.1, max_iter=2000,
                random_state=QUERY_ONLY_SEED,
            )),
        ]), False),
        "embedding_mlp": ("embedding", Pipeline([
            ("scale", StandardScaler()),
            ("classifier", MLPClassifier(
                hidden_layer_sizes=(64,), alpha=1.0, max_iter=500,
                early_stopping=True, validation_fraction=0.15,
                n_iter_no_change=25, random_state=QUERY_ONLY_SEED,
            )),
        ]), True),
    }


def balance_training_fold(x, y, seed):
    y = np.asarray(y)
    rng = np.random.default_rng(seed)
    groups = [np.flatnonzero(y == label) for label in np.unique(y)]
    target_size = max(map(len, groups))
    indices = np.concatenate([
        np.concatenate([group, rng.choice(group, target_size - len(group), replace=True)])
        if len(group) < target_size else group
        for group in groups
    ])
    rng.shuffle(indices)
    return (x.iloc[indices] if hasattr(x, "iloc") else x[indices]), y[indices]


def benchmark_input(data, input_type, linguistic_columns, embeddings):
    if input_type == "text":
        return data["query_text"]
    if input_type == "linguistic":
        return data[linguistic_columns]
    return embeddings


def evaluate_query_only_benchmark(data, embeddings):
    y = data["rerank_beneficial"].to_numpy()
    linguistic_columns = list(query_features("").keys())
    split_sets = {
        "stratified_5_fold": list(StratifiedKFold(
            5, shuffle=True, random_state=QUERY_ONLY_SEED
        ).split(data, y)),
        "leave_one_dataset_out": list(GroupKFold(
            data["dataset"].nunique()
        ).split(data, y, groups=data["dataset"])),
    }
    summary_rows, prediction_frames = [], []
    for model_name, (input_type, template, oversample) in query_only_model_specs().items():
        x = benchmark_input(data, input_type, linguistic_columns, embeddings)
        for evaluation, splits in split_sets.items():
            probabilities = np.zeros(len(data))
            for fold, (train_idx, test_idx) in enumerate(splits):
                x_train = x.iloc[train_idx] if hasattr(x, "iloc") else x[train_idx]
                x_test = x.iloc[test_idx] if hasattr(x, "iloc") else x[test_idx]
                y_train = y[train_idx]
                if oversample:
                    x_train, y_train = balance_training_fold(
                        x_train, y_train, QUERY_ONLY_SEED + fold
                    )
                model = clone(template).fit(x_train, y_train)
                probabilities[test_idx] = model.predict_proba(x_test)[:, 1]
            metrics = classifier_evaluation_summary(data, probabilities, evaluation)
            metrics.update({"model": model_name, "input": input_type})
            summary_rows.append(metrics)
            fold_predictions = data[[
                "dataset", "query_id", "query_text",
                "delta_nDCG@10", "rerank_beneficial",
            ]].copy()
            fold_predictions["model"] = model_name
            fold_predictions["evaluation"] = evaluation
            fold_predictions["rerank_probability"] = probabilities
            fold_predictions["rerank_predicted"] = (probabilities >= 0.5).astype(int)
            prediction_frames.append(fold_predictions)
    return pd.DataFrame(summary_rows), pd.concat(prediction_frames, ignore_index=True)


def run_query_only_benchmark(rankings, metric_deltas, output_dir="."):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    data = build_query_only_data(rankings, metric_deltas)
    print(f"Gerando embeddings para {len(data)} queries...")
    encoder = SentenceTransformer(QUERY_ONLY_EMBEDDING_MODEL)
    embeddings = encoder.encode(
        data["query_text"].tolist(), batch_size=64, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )
    summary, predictions = evaluate_query_only_benchmark(data, embeddings)
    data.to_csv(output_dir / "query_only_classifier_dataset.csv", index=False)
    summary.to_csv(output_dir / "query_only_model_benchmark.csv", index=False)
    predictions.to_csv(output_dir / "query_only_model_predictions.csv", index=False)
    np.save(output_dir / "query_only_embeddings.npy", embeddings)
    return {"dataset": data, "summary": summary, "predictions": predictions}


In [ ]:
query_only_benchmark = run_query_only_benchmark(
    rankings=rankings_all,
    metric_deltas=query_deltas,
    output_dir=".",
)

query_only_summary = query_only_benchmark["summary"].sort_values(
    ["evaluation", "roc_auc"], ascending=[True, False]
)
display(query_only_summary.round(4))

plt.figure(figsize=(11, 6))
sns.barplot(
    data=query_only_summary,
    x="roc_auc", y="model", hue="evaluation",
)
plt.axvline(0.5, color="black", linestyle="--", linewidth=1)
plt.xlim(0, 1)
plt.title("Benchmark query-only para decidir quando rerankear")
plt.tight_layout()
plt.savefig("query_only_model_benchmark.png", dpi=160, bbox_inches="tight")
plt.show()


## Síntese dos principais resultados

Considerando os resultados desta execução:

| Dataset | BM25 nDCG@10 | BM25 + BGE nDCG@10 | Δ nDCG@10 | Wilcoxon |
|---|---:|---:|---:|---:|
| SciFact | 0,652 | 0,695 | +0,043 | `p = 0,00047` |
| TREC-COVID | 0,577 | 0,643 | +0,067 | `p = 0,00878` |
| NF-Corpus | 0,306 | 0,309 | +0,002 | `p = 0,95077` |

Principais interpretações:

- **SciFact:** houve melhora consistente de nDCG@10 e MRR@10. O Candidate Recall@100 de 0,873 indica boa cobertura inicial do BM25, oferecendo ao reranker espaço para reorganizar evidências relevantes.
- **TREC-COVID:** apresentou o maior ganho médio de nDCG@10 e MRR@10. O Recall absoluto permanece baixo porque cada tópico possui, em média, centenas de documentos julgados relevantes, enquanto a avaliação considera apenas as primeiras posições.
- **NF-Corpus:** o nDCG@10 ficou praticamente estável e o Recall@10 caiu de 0,152 para 0,143. O ganho de nDCG não foi estatisticamente sustentado, portanto o resultado deve ser descrito como inconclusivo ou dependente da query.
- **Custo:** o reranker aumentou a latência média em todos os datasets. A razão relativa parece especialmente alta nos corpora menores porque o BM25 em memória é muito rápido.
- **RRF:** a fusão entre sinais lexicais e neurais produziu os melhores valores de nDCG@10 em NF-Corpus e TREC-COVID, mas é uma análise pós-hoc e seu tempo adicional não foi medido separadamente.
- **Política seletiva:** o classificador com features pré-reranqueamento obteve AUC 0,714 na validação estratificada, mas AUC 0,474 em *leave-one-dataset-out*. Isso sugere aprendizado de padrões internos dos datasets, com pouca evidência de transferência robusta para um domínio novo.

Em conjunto, os resultados apoiam a hipótese de que o reranqueamento neural pode melhorar significativamente as primeiras posições, mas mostram que seu benefício depende da cobertura inicial, do domínio e do custo aceitável.


## Limitações do experimento

- O reranker só reordena os documentos que vieram do BM25; se um documento relevante não entra nos candidatos iniciais, ele não pode aparecer no ranking final.
- O BGE-Reranker pode ter sido treinado em dados próximos a alguns domínios avaliados, o que pode favorecer certos datasets.
- O custo computacional do reranker é maior que o do BM25, especialmente quando `BM25_TOP_N` cresce.
- Os datasets avaliados são em inglês e têm forte viés científico/biomédico; os resultados não necessariamente transferem para outro idioma ou domínio.
- A visualização 3D é ilustrativa: PCA reduz embeddings de alta dimensão para três dimensões e pode distorcer distâncias.
- Bons resultados em BEIR não garantem automaticamente bom desempenho em uma base real de produção, com documentos mais ruidosos, longos ou desatualizados.


## Referências

[1] Thakur, N. et al. **BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models**. NeurIPS Datasets and Benchmarks, 2021. https://arxiv.org/abs/2104.08663

[2] Wadden, D. et al. **Fact or Fiction: Verifying Scientific Claims**. EMNLP, 2020. https://arxiv.org/abs/2004.14974

[3] Roberts, K. et al. **Searching for Scientific Evidence in a Pandemic: An Overview of TREC-COVID**. Journal of Biomedical Informatics, 2021. https://arxiv.org/abs/2104.09632

[4] Boteva, V.; Gholipour, D.; Sokolov, A.; Riezler, S. **A Full-Text Learning to Rank Dataset for Medical Information Retrieval**. ECIR, 2016. https://doi.org/10.1007/978-3-319-30671-1_58

[5] Robertson, S.; Zaragoza, H. **The Probabilistic Relevance Framework: BM25 and Beyond**. Foundations and Trends in Information Retrieval, 2009. https://doi.org/10.1561/1500000019

[6] Xiao, S. et al. **C-Pack: Packed Resources for General Chinese Embeddings**. 2023. https://arxiv.org/abs/2309.07597  
Modelo utilizado: https://huggingface.co/BAAI/bge-reranker-base

[7] Cormack, G. V.; Clarke, C. L. A.; Buettcher, S. **Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods**. SIGIR, 2009. https://doi.org/10.1145/1571941.1572114

[8] MacAvaney, S. et al. **Simplified Data Wrangling with ir_datasets**. SIGIR, 2021. https://ir-datasets.com/


### Organizar outputs na pasta `results/`

Esta célula cria a pasta `results/` e move para lá todos os artefatos gerados pelo notebook. Se a pasta já tiver arquivos antigos do relatório, eles são substituídos antes de salvar a nova execução.


In [ ]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

OUTPUT_EXTENSIONS = {".csv", ".json", ".png", ".html", ".md"}

RESULT_SUBDIR_RULES = [
    ("rankings", lambda p: p.name.startswith("rankings") or p.name.startswith("query_rankings")),
    ("qualitative", lambda p: "qualitative" in p.name or "failure" in p.name or "rank_overlap" in p.name or "relevant_documents" in p.name),
    ("visualizations", lambda p: "visualization" in p.name or p.suffix.lower() in {".png", ".html"}),
    ("metrics", lambda p: p.name.startswith("results") or "summary" in p.name or "metrics" in p.name or "deltas" in p.name or "precision_recall" in p.name),
    ("reports", lambda p: p.suffix.lower() == ".md" or "report" in p.name or "relatorio" in p.name.lower()),
]


def output_subdir_for(file_path):
    for subdir_name, predicate in RESULT_SUBDIR_RULES:
        if predicate(file_path):
            return subdir_name
    return "other"

# Limpa artefatos antigos dentro de results/ para evitar mistura entre execuções.
for old_path in sorted(RESULTS_DIR.rglob("*"), reverse=True):
    if old_path.is_file() and old_path.suffix.lower() in OUTPUT_EXTENSIONS:
        old_path.unlink()
    elif old_path.is_dir() and not any(old_path.iterdir()):
        old_path.rmdir()

moved_files = []
for file_path in Path(".").iterdir():
    if not file_path.is_file():
        continue
    if file_path.suffix.lower() not in OUTPUT_EXTENSIONS:
        continue
    if file_path.name.startswith("results_outputs"):
        continue

    subdir = output_subdir_for(file_path)
    destination_dir = RESULTS_DIR / subdir
    destination_dir.mkdir(parents=True, exist_ok=True)

    destination = destination_dir / file_path.name
    if destination.exists():
        destination.unlink()
    file_path.replace(destination)
    moved_files.append(str(destination.relative_to(RESULTS_DIR)))

moved_files = sorted(moved_files)
print(f"Arquivos organizados em {RESULTS_DIR}/: {len(moved_files)}")
for file_name in moved_files:
    print("-", file_name)


### Lista de arquivos gerados

Esta célula apenas imprime os nomes dos CSVs, PNGs e HTMLs que foram salvos durante a execução do notebook.


In [ ]:
RESULTS_DIR = Path("results")
if RESULTS_DIR.exists():
    result_files = sorted(path for path in RESULTS_DIR.rglob("*") if path.is_file())
    print(f"Arquivos salvos em {RESULTS_DIR}/ ({len(result_files)} arquivos):")
    for file_path in result_files:
        print("-", file_path.relative_to(RESULTS_DIR))
else:
    print("A pasta results/ ainda não existe. Execute a célula 'Organizar outputs na pasta results/' primeiro.")


### Download dos resultados no Colab

Esta célula usa `google.colab.files` para baixar os principais arquivos gerados: resultados consolidados, gráficos estáticos e visualizações interativas em HTML.


In [ ]:
# Opcional: compactar e baixar todos os resultados do Colab.
from google.colab import files

RESULTS_DIR = Path("results")
if not RESULTS_DIR.exists():
    raise FileNotFoundError("A pasta results/ não existe. Execute a célula 'Organizar outputs na pasta results/' primeiro.")

zip_path = shutil.make_archive("results_outputs", "zip", RESULTS_DIR)
files.download(zip_path)
print(f"Arquivo compactado para download: {zip_path}")
